# 第 15 章　本地 AI 综合实战

> **本章定位**：把第 14 章的 8 项前置技能串成两条完整流水线——**案例一：AI 图片标注**（视觉模型批量标注交通场景图片，输出 JSON / CSV），**案例二：自然语言处理**（文本模型把交通事件自由文本抽取成结构化分类结果）。两个案例都跑在本机 Ollama 上，适配 RTX 3050，不依赖任何外部在线 API。
>
> **知识点范围**：本地 AI 应用架构与离线边界、提示词与 JSON Schema、视觉模型 `images` 字段、文本预清洗与分类、结果校验 / 重试 / 降级、显存档位与自动降级链。
>
> **环境准备**：见本章「零、环境配置与验收标准」与第 14 章 14.9 部署全流程。当前教学机状态为 **Ollama 尚未部署**，因此本章从零写起，并同时提供 8GB 与 4GB / 6GB 两套配置。
>
> **本章约定（重要）**：
> - 所有主程序**默认走「离线演示模式」**：不联网、不启动 Ollama、不占用显卡，用确定性桩数据跑通整条流水线，输出在任何机器上逐字一致，可当堂校对；
> - 加 `--live` 才真实调用本机 Ollama；服务未就绪时**自动降级**回演示模式并明确打印提示，绝不会卡死；
> - 真实调用只连 `127.0.0.1:11434`，无账号、无 API Key、无外部在线推理；
> - **首次 `ollama pull` 需要联网下载模型权重，之后可断网运行**——「离线运行」不等于「下载阶段也离线」。
>
> **编号约定**：本章无独立场景导入，知识点号与节号一致（15.1—15.6）；第 1—13 章与 5A 节采用「知识点号 = 节号 − 1」。跨章引用一律以「知识点 X.Y」为准。

> **先修知识**：第 01—14 章全部语法与 14.9 的部署结论。默认演示模式无额外前置；`--live` 需 NVIDIA 驱动、本机 Ollama 服务（`127.0.0.1:11434`）与已下载的模型权重。
>
> **学习目标**：完成本章后，你能够：
>
> 1. 直接运行两个案例主程序，逐字核对默认演示模式输出与讲义「运行结果」一致；
> 2. 说明演示模式与 `--live` 两条路径的差别，指出服务不可用时程序降级到哪一步；
> 3. 改写 1 个提示词与配套 JSON Schema，使返回字段通过校验，写出修改前后的字段差异；
> 4. 用校验、重试、降级逻辑处理 1 条非法返回，指出被跳过的记录与重试次数；
> 5. 按 15.0.4 的 5 类验收逐项执行，记录本机显存占用与连续 3 次重跑的字段一致性结论。

---

## 本章学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 零 | 环境配置与验收标准 | 15.0 | RTX 3050 + Ollama 从零部署，双显存档位 |
| 第一部分 | 知识讲解 | 知识点 15.1 — 15.6 | 架构 / 提示词 / 视觉 / 文本 / 容错 / 显存 |
| 第二部分 | 代码示范 | 案例 15-1、案例 15-2 | AI 图片标注 / 交通事件 NLP |
| 第三部分 | 布置练习 | 练习 15-1、练习 15-2 | 置信度过滤 / 分布与准确率统计 |
| 第四部分 | 习题答案 | 答案 15-1、答案 15-2 | 完整参考代码与解析 |
| 验收 | 效果验证方法 | 15.0.4 | 五类验证，区分「已验证」与「需本机验证」 |

**知识点与代码对照表**：

| 知识点 | 名称 | 讲解位置 | 代码示范 | 对应练习 |
|---|---|---|---|---|
| 15.1 | 本地 AI 应用架构与离线边界 | 15.1 | 案例 15-1、15-2 | — |
| 15.2 | 提示词设计与 JSON Schema | 15.2 | 案例 15-1、15-2 | — |
| 15.3 | 视觉模型 `images` 字段与批量标注 | 15.3 | 案例 15-1 | 练习 15-1 |
| 15.4 | 文本预清洗与分类抽取 | 15.4 | 案例 15-2 | 练习 15-2 |
| 15.5 | 结果校验、重试与降级 | 15.5 | 案例 15-1、15-2 | 练习 15-1、15-2 |
| 15.6 | RTX 3050 显存管理与自动降级 | 15.6 | `ai_common.py` | — |

## 教学提示（教师用）

- **建议课时**：12 学时（6 次课）。环境部署 2 学时、知识讲解 4 学时、两个案例 4 学时、练习与验收 2 学时。
- **务必先用默认模式上课**：直接运行 `python3 case1_AI图片标注.py`，不装 Ollama 也能得到完整输出，学生先把流水线看懂；确认无误后再加 `--live` 上真模型。**先工程、后智能**，否则学生只会盯着模型答案，看不懂校验与容错。
- **本环境验证边界（请如实告知学生）**：课件中的全部示例输出已在无 GPU、无 Ollama 的环境下**实跑采集并逐字校验**；真实 GPU 推理的准确率、耗时与显存占用**必须在学生本机按 15.0.4 验收**，教师不应替学生预设结果。
- **学生常见卡点**：把「离线演示」当成「真的在跑模型」；以为加了 `format` 就不用校验；把 `images` 写成字符串而非列表；把 `message.content` 当字典；`--vram 4` 拼写漏空格；服务未启时不知道已自动降级。
- **课堂活动建议**：分组用同一批图片分别跑 8GB 档与 4GB 档，对比 `nvidia-smi` 显存占用与首次推理耗时，再换各自的提示词看「已知答案样本命中数」变化，把「提示词—质量」的因果关系讲透。

---

## 15.0 环境配置与验收标准

### 15.0.1 完整部署步骤（当前教学机 Ollama 尚未部署）

> 详细命令与逐屏现象见**第 14 章 14.9 本机部署全流程**，此处给出本章专用的精简清单与验收标准。

| 步骤 | 命令 / 动作 | 验收标准 |
|---|---|---|
| 1 | `nvidia-smi` | 能看到 RTX 3050 与显存总量（8192 / 4096 / 6144 MiB） |
| 2 | 安装 Ollama，`ollama --version` | 能打印版本号 |
| 3 | `ollama serve`（保持终端窗口） | `curl http://127.0.0.1:11434/api/version` 返回 JSON |
| 4 | 按档位 `ollama pull`（**唯一需联网阶段**） | `ollama list` 能列出模型名与体积 |
| 5 | `ollama run qwen2.5:3b` | 中文问答正常，`/bye` 退出 |
| 6 | 断网后重跑步骤 3、5 | 仍可用，证明推理阶段不访问外网 |
| 7 | 填「本机配置卡」 | 6 个字段齐全，见 demo15-6b |

**8GB 档（台式 RTX 3050 8GB，推荐）**：

```bash
ollama pull gemma3:4b        # 案例一：视觉图片标注
ollama pull qwen2.5:3b       # 案例二：文本 NLP
ollama pull moondream        # 视觉轻量兜底
```

**4GB / 6GB 档（笔记本 RTX 3050，降级）**：

```bash
ollama pull moondream        # 案例一主用
ollama pull qwen2.5:3b       # 案例二
```

**不要选 11B / 13B 视觉模型**：在 3050 上会超显存或退到 CPU，单次推理数分钟，课堂无法完成。

### 15.0.2 本章目录与文件

```text
15本地AI综合实战/
├── 案例与练习.md                    本章讲义（唯一内容源）
├── 15_本地AI综合实战.ipynb          投屏课件（由脚本生成）
├── ai_common.py                     共享模块：模型选择 / HTTP / 校验 / 落盘
├── make_sample_images.py            纯标准库生成 4 张交通示意 PNG
├── sample_images/                   4 张示例图片（开箱即用，可换成自己的照片）
├── events_sample.txt                5 条交通事件文本
├── case1_AI图片标注.py              案例 15-1 主程序
├── case2_交通事件NLP.py             案例 15-2 主程序
├── practice1_置信度过滤.py          练习 15-1
├── practice2_事件分布统计.py        练习 15-2
├── practice1_置信度过滤_答案.py     参考答案
├── practice2_事件分布统计_答案.py   参考答案
└── output/                          运行产物（labels.json / labels.csv / nlp_*，不入库）
```

### 15.0.3 运行方式（三种）

```bash
# ① 默认：离线演示模式，无需 Ollama / GPU，课堂统一校对用
python3 case1_AI图片标注.py
python3 case2_交通事件NLP.py

# ② 真实调用本机 Ollama（8GB 档）
python3 case1_AI图片标注.py --live
python3 case2_交通事件NLP.py --live

# ③ 真实调用 + 4GB/6GB 显存档位（自动换 moondream）
python3 case1_AI图片标注.py --live --vram 4
python3 case2_交通事件NLP.py --live --vram 4
```

> `--vram 4` 只影响视觉模型：案例二始终用 `qwen2.5:3b`（约 1.9 GB，两种档位都够）。服务未启动时 `--live` 会自动降级为演示模式并打印原因，不会报错退出。

---

## 15.0.4 效果验证方法（五类验收）

> **验证边界声明**：本课件在开发环境中已完成的是——全部示例代码的**语法检查、离线演示模式实跑、输出与文档逐字比对**（由 `verify_demos.py` 全量校验）、降级路径模拟、输出文件结构校验。由于开发环境无 RTX 3050 显卡且未部署 Ollama，**真实 GPU 推理的准确率、耗时与显存占用无法在此实测**，以下第 3—5 类必须在学生本机完成，教师不应预设结果。

### 验证 1　离线演示输出逐字比对（已验证 · 无需显卡）

```bash
cd 15本地AI综合实战
python3 case1_AI图片标注.py
python3 case2_交通事件NLP.py
```

把终端输出与讲义「运行结果」逐行对照，应完全一致。两个程序的输出已纳入 `verify_demos.py` 自动校验。

### 验证 2　输出文件结构校验（已验证 · 无需显卡）

```python
# 在 15本地AI综合实战/ 目录下执行
from pathlib import Path
import json, csv

labels = json.loads(Path("output/labels.json").read_text(encoding="utf-8"))
print("labels.json 条数：", len(labels))
print("字段齐全：", all(set(["file", "scene", "congestion", "risk"]) <= set(r) for r in labels))
with open("output/nlp_report.csv", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))
print("nlp_report.csv 条数：", len(rows), "| 列名：", list(rows[0].keys()))
```

演示模式下期望：JSON 3 条、字段齐全 `True`、CSV 5 条、列名含 `time / road / event_type / severity / speed`。

### 验证 3　已知答案样本核对（需本机 · 真实模型）

| 案例 | 标准答案 | 验收动作 |
|---|---|---|
| 案例 15-1 | `road_clear→畅通`、`road_jam→拥堵`、`road_incident→高` | 看「已知答案样本： 3 条 \| 命中： N」，记录 N |
| 案例 15-2 | `TRUTH` 中 5 条事件的人工分类 | 看「准确率」，记录数值 |

改提示词或换模型后**重跑同一批样本**，命中数不应下降。若下降，说明改动引入了退化。

### 验证 4　断网自检（需本机 · 证明离线边界）

1. 先完成 `ollama pull`，确认 `ollama list` 有模型；
2. 断开网络；
3. 保持 `ollama serve`，执行 `python3 case1_AI图片标注.py --live`；
4. 首行应显示「真实调用」，且能正常产出结果文件；
5. 若失败，检查是否误用了 `localhost` 解析到 IPv6，统一改 `127.0.0.1`。

### 验证 5　显存占用与双档位核对（需本机 · RTX 3050）

| 检查项 | 命令 | 记录 |
|---|---|---|
| 显存总量 | `nvidia-smi` | ____ MiB |
| 已加载模型 | `ollama ps` | ____ |
| 推理峰值显存 | 推理时另开终端 `nvidia-smi` | ____ MiB |
| 8GB 档模型 | `python3 case1_AI图片标注.py --live` 首行 | ____ |
| 4GB 档模型 | `python3 case1_AI图片标注.py --live --vram 4` 首行 | ____ |
| 首次 / 第二次耗时 | 秒表或 `time` 命令 | ____ / ____ 秒 |

> 若 4GB 档仍报显存不足，关掉浏览器等占显存程序，或把图片长边缩到 512 px 后重试。

### 降级路径验证（无需显卡）

```bash
python3 case1_AI图片标注.py --live     # 在 Ollama 未启动时执行
```

应打印「本机 Ollama 未就绪……自动降级为离线演示模式」，随后仍能完整产出结果文件，不抛异常、不卡死。

---

## 15.0.5 配套文件：共享模块与示例素材

### 共享模块 `ai_common.py`

两个案例共用同一套「选模型 → 发请求 → 校验 → 落盘 → 汇总」代码，对应第 10 章「模块与程序包」和第 6 章「函数封装」。单独成文件的好处：改一处（例如超时时间）两个案例同时生效，也便于单独测试。

In [ ]:
# -*- coding: utf-8 -*-
"""第 15 章共享工具模块：本地 Ollama 调用、模型选择、结果校验与落盘。

两个实战案例共用本模块，对应第 10 章「模块与程序包」的组织方式。

设计原则：
1. 只依赖 Python 标准库，不装任何第三方包；
2. 真实调用只走 127.0.0.1:11434，不访问任何外部在线 API；
3. 模型返回一律「先解析、再校验、后使用」，坏数据记日志不伪造默认值。
"""
import base64
import csv
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE = "http://127.0.0.1:11434"
TIMEOUT = 120                       # 首次加载模型较慢，给出足够超时

# RTX 3050 双档位：8GB 台式 / 4GB·6GB 笔记本
PROFILES = {
    8: {"vision": "gemma3:4b", "text": "qwen2.5:3b"},
    4: {"vision": "moondream", "text": "qwen2.5:3b"},
}


def select_model(kind, vram_gb=8):
    """按显存容量选档位与模型，返回 (档位, 模型名)。"""
    key = 8 if vram_gb >= 8 else 4
    return key, PROFILES[key][kind]


def check_service(timeout=2):
    """本机服务连通性自检，返回 (是否连通, 说明)。"""
    try:
        with urlopen(BASE + "/api/version", timeout=timeout) as resp:
            return True, "HTTP %d" % resp.status
    except (HTTPError, URLError) as e:
        return False, "服务不可达：%s" % e
    except Exception as e:
        return False, "未知错误：%s" % e


def image_to_b64(path):
    """图片 -> 纯 base64 字符串（Ollama 的 images 字段只收载荷，不要 data URL 前缀）。"""
    return base64.b64encode(Path(path).read_bytes()).decode("ascii")


def post_chat(model, prompt, schema, images=None, timeout=TIMEOUT):
    """向本机 /api/chat 发一次 POST，返回模型输出的 content 字符串。"""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "format": schema,                  # 语法约束：只保证 JSON 可解析
        "options": {"temperature": 0},     # 降低随机性，便于课堂复现
    }
    if images:
        payload["messages"][0]["images"] = images
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = Request(BASE + "/api/chat", data=body,
                  headers={"Content-Type": "application/json"}, method="POST")
    with urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))["message"]["content"]


def validate(content, required, enums=None):
    """解析 + 语义校验，返回 (数据, 说明)；任何一环失败都返回 (None, 原因)。"""
    try:
        data = json.loads(content)
    except json.JSONDecodeError as e:
        return None, "JSON 解析失败：" + e.msg
    if not isinstance(data, dict):
        return None, "返回不是 JSON 对象"
    missing = [k for k in required if k not in data]
    if missing:
        return None, "缺字段：" + "、".join(missing)
    for key, allowed in (enums or {}).items():
        if data.get(key) not in allowed:
            return None, "%s 取值非法：%s" % (key, data.get(key))
    return data, "校验通过"


def save_json(rows, path):
    """结构化结果落盘为 UTF-8 JSON。"""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    return p


def save_csv(rows, path, fields):
    """同一批结果另存 CSV，utf-8-sig 让 Excel 打开中文不乱码。"""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, lineterminator="\n")
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in fields})
    return p


def summary(ok, total, retries, mode, model, vram_key):
    """统一汇总输出；不含耗时等随机器变化的值，便于当堂逐字校对。"""
    print("—— 运行汇总 ——")
    print("运行模式：", mode, "| 显存档位：", vram_key, "GB | 模型：", model)
    print("处理条数：", total, "| 合格：", ok, "| 重试：", retries)
    print("合格率：", round(ok / total * 100, 1) if total else 0.0, "%")
    print("服务地址： 127.0.0.1:11434（仅本机）| 外部在线 API： 无")

### 示例图片生成 `make_sample_images.py`

用 `struct` + `zlib` 手写 PNG 字节，**不依赖 Pillow、不联网下载素材**。`road_foggy.png` 是低对比度图，专门用来触发案例 15-1 的「模型未按格式输出」容错分支。

In [ ]:
# -*- coding: utf-8 -*-
"""用纯标准库生成 4 张交通场景示意 PNG，供案例 15-1 开箱演示。

不需要联网下载素材，也不需要 Pillow。
真实教学时，把学生自己拍的路口照片放进 sample_images/ 即可覆盖本示意图。

用法：
  python3 make_sample_images.py                 # 生成到 ./sample_images
  python3 make_sample_images.py 指定目录        # 生成到指定目录
"""
import struct
import sys
import zlib
from pathlib import Path


def make_png(width, height, painter):
    """按 painter(x, y) -> (r, g, b) 逐像素生成 PNG 字节（8bit RGB）。"""
    raw = bytearray()
    for y in range(height):
        raw.append(0)                         # PNG 每行的 filter type：0 = None
        for x in range(width):
            raw.extend(painter(x, y))

    def chunk(tag, data):
        return (struct.pack(">I", len(data)) + tag + data +
                struct.pack(">I", zlib.crc32(tag + data) & 0xffffffff))

    ihdr = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    return (b"\x89PNG\r\n\x1a\n" +
            chunk(b"IHDR", ihdr) +
            chunk(b"IDAT", zlib.compress(bytes(raw), 9)) +
            chunk(b"IEND", b""))


def clear_scene(x, y):
    """畅通：绿灯 + 一辆车 + 车道虚线。"""
    if (x - 48) ** 2 + (y - 12) ** 2 <= 64:
        return (60, 205, 95)
    if 30 <= y < 33 and (x // 8) % 2 == 0:
        return (238, 238, 238)
    if 20 <= x < 28 and 38 <= y < 54:
        return (45, 115, 225)
    return (72, 76, 82)


def jam_scene(x, y):
    """拥堵：同一车道密集排了 6 辆车。"""
    for i in range(6):
        if 6 + i * 9 <= x < 12 + i * 9 and 22 <= y < 42:
            return (200 - i * 12, 80 + i * 10, 60 + i * 15)
    for i in range(5):
        if 8 + i * 11 <= x < 15 + i * 11 and 46 <= y < 62:
            return (60 + i * 18, 120, 200 - i * 14)
    return (72, 76, 82)


def incident_scene(x, y):
    """事故：两车相接 + 黄色警示块。"""
    if 10 <= x < 28 and 26 <= y < 42:
        return (215, 70, 60)
    if 28 <= x < 44 and 28 <= y < 44:
        return (55, 110, 215)
    if 50 <= x < 60 and 8 <= y < 18:
        return (240, 200, 50)
    return (72, 76, 82)


def foggy_scene(x, y):
    """雾天：整体低对比度，用于演示「模型未按格式输出」的容错分支。"""
    v = 140 + ((x + y) // 8) % 4 * 3
    if 26 <= x < 38 and 28 <= y < 40:
        return (v - 8, v - 8, v - 8)
    return (v, v, v)


SCENES = {
    "road_clear": clear_scene,
    "road_jam": jam_scene,
    "road_incident": incident_scene,
    "road_foggy": foggy_scene,
}


def build(target):
    """在 target 目录生成 4 张 64x64 示意 PNG，返回排序后的文件名列表。"""
    root = Path(target)
    root.mkdir(parents=True, exist_ok=True)
    made = []
    for name, painter in SCENES.items():
        p = root / (name + ".png")
        p.write_bytes(make_png(64, 64, painter))
        made.append(p.name)
    return sorted(made)


if __name__ == "__main__":
    out = sys.argv[1] if len(sys.argv) > 1 else "sample_images"
    names = build(out)
    print("已生成示意图片：", len(names), "张 →", out)
    for n in names:
        print(" ", n, (Path(out) / n).stat().st_size, "字节")

### 运行结果

```text
已生成示意图片： 4 张 → sample_images
  road_clear.png 207 字节
  road_foggy.png 348 字节
  road_incident.png 168 字节
  road_jam.png 225 字节
```

> 字节数随 Python / zlib 版本略有差异，不影响图片内容与后续流程。`sample_images/` 已随课件提供，平时不必重复运行本脚本；想换成学生自拍的路口照片时，直接把 `.png` / `.jpg` 放进该目录即可。

### 示例文本 `events_sample.txt`

案例 15-2 的输入，5 条半结构化交通事件日志（含车牌、时间、路段、车速）。保存为 UTF-8 纯文本：

```text
08:12 京A12345 机场北 发生追尾，两车轻微剐蹭，车速降至 25km/h
09:05 京B67890 京开高速 早高峰缓行，平均车速 42km/h
10:30 京C11122 机场南 雨天路滑，一辆小车失控撞护栏，车速 18km/h
14:20 京D33344 京港澳 道路施工占一车道，通行缓慢，车速 38km/h
17:45 京E55566 机场北 车流恢复正常，平均车速 65km/h
```

---

# 第一部分　知识讲解

## 15.1 知识点 15.1　本地 AI 应用架构与离线边界

### 15.1.1 核心概念

**「本地 AI 应用」不是把模型写进 Python，而是 Python 与一个本机服务分工**：

| 角色 | 负责 | 技术 |
|---|---|---|
| Python 程序 | 找文件、转码、组请求、校验、落盘 | 标准库（第 14 章） |
| Ollama 服务 | 加载权重、在 GPU 上推理、返回结果 | `127.0.0.1:11434` |
| 模型权重 | 决定「看得懂什么」 | 本地磁盘，首次 `pull` 下载 |

一次调用跨过 7 个环节，其中只有「模型推理」用到显卡，其余全是普通 Python 数据处理。这也解释了为什么本章默认演示模式能跑：桩数据替换掉第 5 步，另外 6 步完全一样。

**离线边界的三条硬规则**：

1. 服务地址只能是 `127.0.0.1` / `localhost`，端口 `11434`；
2. 请求不携带任何账号、Token、API Key；
3. 权重下载（`ollama pull`）需联网，推理阶段可断网。

**原理补充（网络视角）**：`127.0.0.1` 是回环地址，数据包在操作系统内核的虚拟网卡上直接回送，不经过物理网卡、不经过路由器、不产生外网流量。因此断网后 `curl 127.0.0.1:11434` 依然成功。用 `urllib.parse.urlparse()` 检查 host 是否在白名单里，是防止「代码里混进一个云服务地址」最便宜的自检手段——这不是防攻击，而是防手误。

### 15.1.2 语法要点

```python
from urllib.parse import urlparse
from ai_common import BASE, check_service, post_chat

host = urlparse(BASE).hostname          # '127.0.0.1'
ok, info = check_service()              # (True, 'HTTP 200') 或 (False, 原因)
content = post_chat(model, prompt, schema, images=[b64])
```

- 主程序启动时**先 `check_service()` 再决定是否 `--live`**，避免请求发出去才失败；
- 所有 URL 常量集中在 `ai_common.BASE`，不要在业务代码里散落字符串；
- `stream: False` 表示一次性拿完整回答，教学场景不涉及流式 NDJSON。

### 15.1.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 把演示模式当真实推理 | 学生以为「模型很准」 | 主程序开头明确打印「运行模式」 |
| 服务地址写成云 API | 产生外网请求、需要密钥 | 统一用 `ai_common.BASE` + host 白名单 |
| `ollama serve` 窗口关掉 | `Connection refused` | 先 `check_service()`，失败自动降级 |
| 认为断网也能 `pull` | 下载报错 | 把「下载」与「推理」分两阶段讲 |
| 把模型塞进 Python 进程 | 内存暴涨、启动极慢 | 模型由 Ollama 服务独立管理 |

### 15.1.4 随堂示例：一次调用的 7 步链路

**核心要点**：把第 14 章的链路表写成可打印的数据结构，并自动算出「有没有外部在线 API 环节」。

In [ ]:
STEPS = [
    ("读取图片", "本地", "pathlib"),
    ("图片转码", "本地", "base64"),
    ("构造提示词", "本地", "str / json"),
    ("POST 请求", "本机服务", "urllib.request"),
    ("模型推理", "本机 GPU", "Ollama"),
    ("解析与校验", "本地", "json / 第9章"),
    ("统计与落盘", "本地", "csv / Counter"),
]
for i, (name, where, tool) in enumerate(STEPS, 1):
    print("步骤", i, "|", name, "|", where, "|", tool)
print("外部在线 API 环节：", [s[0] for s in STEPS if s[1] == "外部"])

**预期输出**：

```text
步骤 1 | 读取图片 | 本地 | pathlib
步骤 2 | 图片转码 | 本地 | base64
步骤 3 | 构造提示词 | 本地 | str / json
步骤 4 | POST 请求 | 本机服务 | urllib.request
步骤 5 | 模型推理 | 本机 GPU | Ollama
步骤 6 | 解析与校验 | 本地 | json / 第9章
步骤 7 | 统计与落盘 | 本地 | csv / Counter
外部在线 API 环节： []
```

**变式代码（迁移练习）**：用 `urlparse` 做地址白名单自检，识别混进来的外部域名。

In [ ]:
from urllib.parse import urlparse

URLS = [
    "http://127.0.0.1:11434/api/chat",
    "http://localhost:11434/api/tags",
    "https://api.example.com/v1/chat",
]
for u in URLS:
    host = urlparse(u).hostname
    local = host in ("127.0.0.1", "localhost")
    print(host, "| 端口", urlparse(u).port, "| 本机：", local)
print("允许调用的地址数：", sum(1 for u in URLS
      if urlparse(u).hostname in ("127.0.0.1", "localhost")))

**预期输出**：

```text
127.0.0.1 | 端口 11434 | 本机： True
localhost | 端口 11434 | 本机： True
api.example.com | 端口 None | 本机： False
允许调用的地址数： 2
```

### 15.1.5 实操步骤（上机跟随）

1. 运行 15.1.4 主示例，对照第 14 章「场景导入」表，说出每一步对应的知识点编号。
2. 在 `STEPS` 里把某一步的 `where` 改成 `"外部"`，观察最后一行是否把它列进「外部在线 API 环节」。
3. 运行变式代码，把 `https://api.example.com/v1/chat` 换成 `http://127.0.0.1:11434/api/version`，确认「允许调用的地址数」变 3。
4. 打开 `ai_common.py`，找到 `BASE`，确认全项目只有这一处定义服务地址。
5. 未部署 Ollama 时执行 `python3 case1_AI图片标注.py --live`，观察自动降级提示，理解「先验服务、后发请求」。

### 15.1.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 分不清「演示模式」和「真实调用」 | 两者输出结构一样 | 看首行「运行模式」，演示模式还会打印「已知答案样本」 |
| `check_service()` 返回 False | 服务未启或端口被占 | 回到 14.9 步骤 3，`lsof -i :11434` |
| 断网后 `curl` 失败 | 用了 `localhost` 且 DNS/IPv6 异常 | 统一写 `127.0.0.1` |
| 担心图片被传到外网 | 把 HTTP 请求误解为公网请求 | 用 14.9 步骤 7 断网自检验证 |
| 想换成云端大模型 | 需求变了 | 明确超出本课程「本地离线」边界，需另行评估数据合规 |

---

## 15.2 知识点 15.2　提示词设计与 JSON Schema 约束

### 15.2.1 核心概念

**提示词（prompt）**是给模型的任务说明书。对本地小模型，三条经验最有效：

1. **角色 + 任务 + 只输出 JSON**，明确禁止解释文字；
2. **枚举写死**：`congestion（畅通/缓行/拥堵三选一）`，别让模型自由发挥；
3. **字段清单与 Schema 一致**，提示词里提到的字段必须出现在 `required` 里。

**JSON Schema** 通过 `/api/chat` 的 `format` 参数传给 Ollama，属于 **grammar-constrained decoding（语法约束解码）**：模型在生成每个 token 时会被限制在「能构成合法 JSON」的候选里。

**原理补充（约束的边界）**：语法约束只管「形式」，不管「含义」。`{"congestion": "严重拥堵"}` 完全合法 JSON，却不在枚举里；`{"congestion": "拥堵"}` 少了 `risk` 字段也能被约束生成（取决于 Schema 是否写 `required`）。因此本章把校验分成两层：**语法层**（`json.loads`）与**语义层**（缺字段 / 枚举 / 类型），后者必须由 `ai_common.validate()` 完成。这是全章最重要的工程观念，也是练习 15-1、15-2 的统计口径为什么容易出错的根源。

### 15.2.2 语法要点

```python
SCHEMA = {
    "type": "object",
    "properties": {
        "congestion": {"type": "string"},
        "risk": {"type": "string"},
        "vehicle_count": {"type": "integer"},
    },
    "required": ["congestion", "risk", "vehicle_count"],
}
payload = {"model": model, "messages": [...], "stream": False,
           "format": SCHEMA, "options": {"temperature": 0}}
```

- `temperature: 0` 用于压低随机性：同一输入连跑 3 次，字段名与取值完全一致视为通过；出现差异时记录差异字段，不得以「模型本来就不稳定」略过；
- `required` 与 Python 侧的 `REQUIRED = SCHEMA["required"]` **共用一份定义**，避免两处不一致；
- 提示词用常量 `PROMPT` 集中管理，重试时只追加一句「请严格只输出 JSON」；
- 枚举校验放在 `ENUMS` 字典里，`validate(content, REQUIRED, ENUMS)` 一次完成。

### 15.2.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 以为 `format` 保证取值合法 | 出现「严重拥堵」等越界值 | 必做枚举校验 |
| 提示词与 `required` 不一致 | 模型稳定漏字段 | 两边同步改 |
| 允许模型输出解释 | 返回 `好的，结果如下：{...}`，`loads` 失败 | 明确「只输出 JSON，不要解释」 |
| 每次重新手写 Schema | 两案例字段不一致 | 常量集中在模块顶部 |
| `temperature` 不设 | 同图两次结果不同 | 设 0，并用结构化字段验收 |

### 15.2.4 随堂示例：Schema 与提示词的一致性检查

In [ ]:
import json

SCHEMA = {
    "type": "object",
    "properties": {
        "congestion": {"type": "string"},
        "risk": {"type": "string"},
        "vehicle_count": {"type": "integer"},
    },
    "required": ["congestion", "risk", "vehicle_count"],
}
PROMPT = "只输出 JSON：congestion（畅通/缓行/拥堵）、risk、vehicle_count"
print("必需字段：", SCHEMA["required"])
print("字段数：", len(SCHEMA["properties"]))
print("提示词长度：", len(PROMPT))
print("schema 序列化长度：", len(json.dumps(SCHEMA, ensure_ascii=False)))

**预期输出**：

```text
必需字段： ['congestion', 'risk', 'vehicle_count']
字段数： 3
提示词长度： 48
schema 序列化长度： 187
```

**变式代码（迁移练习）**：三条模型返回分别代表「合格」「语法对但语义错」「语法错」，直观看到两层的区别。

In [ ]:
import json

answers = [
    '{"congestion": "拥堵", "risk": "高", "vehicle_count": 6}',
    '{"congestion": "严重拥堵", "risk": "高", "vehicle_count": 6}',
    '这张图看起来有点堵。',
]
for i, a in enumerate(answers, 1):
    try:
        d = json.loads(a)
        ok = d["congestion"] in ("畅通", "缓行", "拥堵")
        print("样本", i, "| 语法：可解析 | 语义：", "通过" if ok else "取值非法")
    except json.JSONDecodeError:
        print("样本", i, "| 语法：解析失败 | 语义：未校验")

**预期输出**：

```text
样本 1 | 语法：可解析 | 语义： 通过
样本 2 | 语法：可解析 | 语义： 取值非法
样本 3 | 语法：解析失败 | 语义：未校验
```

### 15.2.5 实操步骤（上机跟随）

1. 运行 15.2.4 主示例，给 `properties` 加一个字段后重跑，确认「字段数」和「schema 序列化长度」同步变化。
2. 把 `required` 删掉一个字段，观察「必需字段」变化，思考提示词里是否也该同步删。
3. 运行变式代码，把样本 2 的 `"严重拥堵"` 改成 `"拥堵"`，确认语义层转为「通过」。
4. 打开 `case1_AI图片标注.py`，核对 `SCHEMA["required"]`、`PROMPT`、`ENUMS` 三处字段是否一一对应。
5. 部署完成后用 `--live` 跑一次，记录模型是否出现过「解释文字 + JSON」的混合输出。

### 15.2.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 加了 `format` 仍返回文本 | 模型能力有限或提示词太弱 | 收紧提示词并重试一次（案例里的 `strict=True`） |
| 枚举值写成「有点堵」 | 提示词没写死候选项 | 明确「三选一」并列出全部取值 |
| 字段偶尔缺失 | `required` 未覆盖该字段 | 补进 `required` + 提示词 |
| 同一提示词两次答案不同 | 采样随机性 | `temperature=0` + 结构化字段验收 |
| Schema 改了但校验没变 | `REQUIRED` 是复制粘贴的 | 统一用 `SCHEMA["required"]` 派生 |

---

## 15.3 知识点 15.3　视觉模型 `images` 字段与批量标注

### 15.3.1 核心概念

**图片标注流水线**（案例 15-1 的完整形态）：

```text
sample_images/*.png → read_bytes → base64 → messages[0].images → /api/chat
                                                    ↓
                          labels.json / labels.csv ← validate ← message.content
```

Ollama 的 `/api/chat` 在**消息对象**里接受 `images` 字段，值是**字符串列表**，每个元素是一张图的纯 base64 载荷（不带 `data:image/...` 前缀）。即使只传一张图也必须写成列表。

**批量标注的三条工程纪律**：

1. `sorted()` 固定顺序，保证输出可逐字校对；
2. 每张图**独立校验、独立记日志**，一张坏图不影响其他图；
3. 结果同时写 JSON（程序读）与 CSV（人 / Excel 读），字段列表 `FIELDS` 统一定义。

**原理补充（为什么一张图失败不该中断整批）**：本地小模型对模糊图、逆光图容易输出非 JSON 文本。若把 `json.loads` 放在循环外或用裸 `loads` 不捕获，第 3 张图失败会让前 2 张的结果也丢掉。案例 15-1 用 `for` + `try` 式的 `validate()` 把失败收敛成 `(None, 原因)`，累计到 `failed` 列表，最后统一打印——这正是第 9 章「容错」在批处理里的标准形态，也是 14.3 练习 14-2 的延续。

### 15.3.2 语法要点

```python
message = {"role": "user", "content": PROMPT, "images": [b64_str]}
content = post_chat(model, PROMPT, SCHEMA, images=[b64_str])
validate(content, REQUIRED, ENUMS)
save_json(rows, OUT / "labels.json")
save_csv(rows, OUT / "labels.csv", FIELDS)
```

- `image_to_b64(path)` 封装 `read_bytes → b64encode → decode("ascii")`；
- 大图先缩放：base64 体积约膨胀 1/3，长边建议 512—768 px（见 demo15-3b）；
- `FIELDS` 的顺序决定 CSV 列顺序，`save_csv` 用 `r.get(k, "")` 容忍缺列；
- 结果里额外加 `file` 与 `model` 两列，方便追溯「哪张图、哪个模型」。

### 15.3.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| `images` 写成字符串 | 接口报错或只识别一部分 | 一律 `[b64]` |
| 加了 data URL 前缀 | `invalid image` | 只传纯载荷 |
| 单图失败中断整批 | 前几张结果丢失 | 每张独立 `validate` + `continue` |
| 忘 `sorted()` | 输出顺序随机 | `sorted(glob("*.png"))` |
| 原图 5 MB 直接编码 | 请求体 6.7 MB，推理慢 | 先压缩 / 缩放 |
| 把失败样本写进结果 | 报表虚高 | 只把 `data` 非 None 的加入 `rows` |

### 15.3.4 随堂示例：构造带图片的消息

In [ ]:
payload = {
    "model": "gemma3:4b",
    "messages": [{"role": "user", "content": "标注这张路口图", "images": ["SU1H"]}],
    "stream": False,
}
m = payload["messages"][0]
print("模型：", payload["model"])
print("images 是列表：", isinstance(m["images"], list))
print("图片数：", len(m["images"]))
print("stream：", payload["stream"])
print("消息字段数：", len(m))

**预期输出**：

```text
模型： gemma3:4b
images 是列表： True
图片数： 1
stream： False
消息字段数： 3
```

**变式代码（迁移练习）**：估算不同原图大小对应的 base64 体积，理解为什么要先压缩。

In [ ]:
sizes = [10 * 1024, 100 * 1024, 500 * 1024]
for n in sizes:
    b64 = 4 * ((n + 2) // 3)
    print(n // 1024, "KB → base64", b64 // 1024, "KB | 膨胀", round(b64 / n, 2), "倍")
print("建议上限：", 300, "KB（长边缩到 512—768 px）")

**预期输出**：

```text
10 KB → base64 13 KB | 膨胀 1.33 倍
100 KB → base64 133 KB | 膨胀 1.33 倍
500 KB → base64 666 KB | 膨胀 1.33 倍
建议上限： 300 KB（长边缩到 512—768 px）
```

### 15.3.5 实操步骤（上机跟随）

1. 运行 15.3.4 主示例，把 `images` 从 `["SU1H"]` 改成 `"SU1H"`，观察「images 是列表」变 `False`。
2. 运行变式代码，代入一张本机真实照片的字节数，估算编码后体积。
3. 执行 `python3 make_sample_images.py`，确认 `sample_images/` 里生成 4 张 PNG。
4. 执行 `python3 case1_AI图片标注.py`，逐行对照「运行结果」；特别看 `road_foggy.png` 如何被记入失败日志而不中断流程。
5. 打开 `output/labels.json` 与 `output/labels.csv`，数一数是不是各 3 条（第 4 张不合格未写入）。

### 15.3.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 所有图都判失败 | 模型名不对 / 传了 data URL 前缀 | 核对 `ollama list`，只传纯载荷 |
| 只有某张图失败 | 图太模糊，模型输出解释文字 | 属正常，看失败日志；必要时重拍或换清晰图 |
| `output/` 里没有文件 | 工作目录不对 | 在项目目录执行，或用绝对路径 |
| CSV 中文乱码 | 用记事本按 ANSI 打开 | `utf-8-sig` 已处理，用 Excel 打开 |
| 推理极慢 | 原图过大 / 显存不足 | 缩图；换 4GB 档 `moondream` |
| 想标自己的照片 | 示意图不合适 | 把 `.png` / `.jpg` 放进 `sample_images/` |

---

## 15.4 知识点 15.4　文本预清洗与分类抽取

### 15.4.1 核心概念

**NLP 案例不做「分词训练」，做的是结构化抽取**：把一句自由文本变成固定字段的字典。流程是「正则先清洗 → 模型判类别 → 校验 → 统计落盘」。

| 阶段 | 工具 | 产出 |
|---|---|---|
| 预清洗 | `re.sub` 脱敏车牌、`re.search` 抽时间/路段/车速 | 干净文本 + 已知字段 |
| 分类 | `qwen2.5:3b` + JSON Schema | `event_type` / `severity` / `summary` |
| 校验 | `validate` | 合格字典或失败原因 |
| 统计 | `Counter` | 事件类型分布 |
| 验收 | 已知答案样本 `TRUTH` | 命中数 / 准确率 |

**为什么先正则再送模型**：时间、路段、车速这类**格式固定**的信息，正则又快又准；把它们先抽出来，模型只需判断「属于哪类事件」这个真正需要语义能力的部分，既省 token 又减少出错。车牌脱敏则属于数据合规：车牌、人名这类个人/车辆标识不该出现在发给模型的文本里，即使模型在本机也一样。

**原理补充（准确率为何要持续监控）**：本地 3B 模型是概率生成，换一批文本、换一句提示词，分类就可能漂移。`TRUTH` 字典是**人工标注的标准答案**，每次运行都比对一次，得到「已知答案样本命中数」。这个数字不追求 100%，它的价值是**发现退化**：提示词改了以后从 5/5 掉到 3/5，就说明改动有问题。真实项目里这叫回归测试集。

### 15.4.2 语法要点

```python
PLATE = re.compile(r"[\u4e00-\u9fa5][A-Z]\d{5}")
clean = PLATE.sub("【车牌】", line)
road = ROAD.search(clean).group() if ROAD.search(clean) else ""
dist = Counter(r["event_type"] for r in rows)
hits = sum(1 for r in rows if TRUTH.get(r["time"]) == r["event_type"])
```

- 预编译 `re.compile()` 放在模块顶层，循环里复用；
- `TRUTH` 用**时间**做键（每条事件时间唯一），比用列表下标更稳；
- `Counter(...).most_common()` 给出按频次排序的分布，练习 15-2 会专门练；
- 落盘前把正则抽出的 `speed` 合并进结果字典，形成一份完整报表。

### 15.4.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 未脱敏就送模型 | 车牌出现在提示词与日志里 | 先 `re.sub` |
| `search` 不判空直接 `.group()` | 某条文本没时间就崩 | `m.group() if m else ""` |
| 用列表下标对齐 `TRUTH` | 增删一条就全部错位 | 用唯一键（时间）关联 |
| 把准确率分母写成命中数 | 恒为 100% | 分母必须是样本总数，见练习 15-2 |
| 用样本数当类别数 | 分布统计失真 | `Counter` 的键数，见练习 15-2 |

### 15.4.4 随堂示例：脱敏与字段预抽取

In [ ]:
import re

line = "08:12 京A12345 机场北 发生追尾，车速降至 25km/h"
clean = re.sub(r"[\u4e00-\u9fa5][A-Z]\d{5}", "【车牌】", line)
print("脱敏后：", clean)
print("时间：", re.search(r"\d{2}:\d{2}", clean).group())
print("车速：", re.search(r"(\d+)\s*km/h", clean).group(1))
print("是否仍含车牌：", bool(re.search(r"[A-Z]\d{5}", clean)))

**预期输出**：

```text
脱敏后： 08:12 【车牌】 机场北 发生追尾，车速降至 25km/h
时间： 08:12
车速： 25
是否仍含车牌： False
```

**变式代码（迁移练习）**：用 `%s` 模板拼提示词，并检查占位符是否残留。

In [ ]:
TEMPLATE = "你是交通事件分类员。只输出 JSON。\n待分类文本：%s"
text = "08:12 【车牌】 机场北 发生追尾"
prompt = TEMPLATE % text
print("模板长度：", len(TEMPLATE))
print("文本长度：", len(text))
print("最终提示词长度：", len(prompt))
print("含占位符残留：", "%s" in prompt)

**预期输出**：

```text
模板长度： 28
文本长度： 19
最终提示词长度： 45
含占位符残留： False
```

### 15.4.5 实操步骤（上机跟随）

1. 运行 15.4.4 主示例，把车牌换成新能源 6 位牌，观察脱敏是否失效，再改模式为 `\d{5,6}`。
2. 在变式代码里把 `%s` 换成 `{}`，思考应改用 `.format()` 还是 f-string。
3. 打开 `events_sample.txt`，逐条预判 `event_type`，写进草稿。
4. 执行 `python3 case2_交通事件NLP.py`，把「事件类型分布」与自己的预判对照。
5. 打开 `output/nlp_report.csv`，确认 `time / road / event_type / severity / speed` 五列齐全，且车牌一栏已脱敏。

### 15.4.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 某条事件 `road` 为空 | 正则名单没收录该路段 | 扩充 `ROAD` 正则的候选项 |
| 分类结果每次不同 | 模型随机性 / 提示词含糊 | `temperature=0` + 写死枚举 |
| 准确率突然下降 | 提示词或模型被改动 | 用 `TRUTH` 回归测试集定位 |
| `events_sample.txt` 读出来乱码 | 文件不是 UTF-8 | 另存为 UTF-8，`read_text(encoding="utf-8")` |
| 车牌仍出现在日志 | 只脱敏了模型输入，没脱敏原始行 | 全流程统一使用 `clean` 文本 |
| 想增加事件类别 | 业务扩展 | 同步改 `SCHEMA`、`ENUMS`、提示词、`TRUTH` 四处 |

---

## 15.5 知识点 15.5　结果校验、重试与降级

### 15.5.1 核心概念

**三层防线**：

| 层 | 检查什么 | 失败处理 |
|---|---|---|
| 语法层 | `json.loads` 能否解析 | 记入失败日志 |
| 语义层 | 必需字段、枚举取值、类型 | 记入失败日志 |
| 服务层 | `check_service()` 是否连通 | 自动降级为演示模式 |

**重试策略**：真实调用失败时，**只重试一次**，并在提示词末尾加一句「请严格只输出 JSON」。不做无限重试——本地小模型对同一张模糊图往往持续失败，重试十次只是浪费显存和时间。

**降级策略**：`--live` 但服务没起 → 打印原因 → 切演示模式 → 继续跑完整流程。这样课堂不会因为一台机器没装好而中断，同时学生能从首行「运行模式」清楚知道自己看的是不是真模型结果。

**原理补充（失败必须留痕）**：`validate()` 返回 `(None, 原因)` 而不是抛异常，是因为「模型返回坏数据」在这个系统里属于**可预期的业务事件**，不是程序错误。把它收敛成数据后，主程序可以统一累计 `failed` 列表、打印清单、写日志。反过来，`urlopen` 抛的 `URLError` 才是真正的运行异常，由 `check_service()` 在服务层提前拦掉。区分「业务失败」与「运行异常」，是第 9 章异常处理的进阶用法。

### 15.5.2 语法要点

```python
data, msg = validate(content, REQUIRED, ENUMS)
if data is None and live:
    content = infer_content(p, model, live, strict=True)   # 收紧提示词
    data, msg = validate(content, REQUIRED, ENUMS)
    retries += 1
if data is None:
    failed.append(p.name)
    continue
```

- 重试**只在 `live` 模式**发生：演示桩是确定性的，重试没有意义；
- `retries` 计数进入 `summary()`，是质量报表的一部分；
- `failed` 列表打印文件名而不是索引，方便直接找到那张图重拍；
- 绝不写 `data.get("level", "拥堵")` 这类兜底默认值——那是伪造数据。

### 15.5.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 无限重试 | 卡死、显存占用不释放 | 限一次 |
| 失败样本填默认值 | 合格率虚高、数据造假 | 记入 `failed`，不写入结果 |
| 演示模式也重试 | 输出多一遍，无意义 | 加 `and live` 条件 |
| 降级后不提示 | 学生误以为是真实结果 | 首行明确打印运行模式 |
| 只打 `Connection refused` 不说明后果 | 学生不知程序还在跑 | 打印「自动降级为离线演示模式」 |

### 15.5.4 随堂示例：三类返回的校验结果

In [ ]:
import json

REQUIRED = ["congestion", "risk", "vehicle_count"]
LEVELS = ["畅通", "缓行", "拥堵"]
answers = [
    '{"congestion": "拥堵", "risk": "高", "vehicle_count": 6}',
    '{"congestion": "拥堵", "risk": "高"}',
    '{"congestion": "拥堵", "risk": "高",',
]
for i, a in enumerate(answers, 1):
    try:
        d = json.loads(a)
    except json.JSONDecodeError as e:
        print("样本", i, "| 结果：解析失败 |", e.msg)
        continue
    missing = [k for k in REQUIRED if k not in d]
    if missing:
        print("样本", i, "| 结果：缺字段 |", missing)
    elif d["congestion"] not in LEVELS:
        print("样本", i, "| 结果：取值非法")
    else:
        print("样本", i, "| 结果：合格 | 车辆数", d["vehicle_count"])

**预期输出**：

```text
样本 1 | 结果：合格 | 车辆数 6
样本 2 | 结果：缺字段 | ['vehicle_count']
样本 3 | 结果：解析失败 | Expecting property name enclosed in double quotes
```

**变式代码（迁移练习）**：把「服务可用」与「是否请求真实调用」两个布尔量组合成降级决策表。

In [ ]:
def decide(service_ok, live_requested):
    if service_ok and live_requested:
        return "真实调用", "gemma3:4b"
    if live_requested:
        return "离线演示（自动降级）", "moondream"
    return "离线演示", "gemma3:4b"

for ok, req in ((True, True), (False, True), (False, False)):
    mode, model = decide(ok, req)
    print("服务可用", ok, "| 请求真实调用", req, "→", mode, "|", model)

**预期输出**：

```text
服务可用 True | 请求真实调用 True → 真实调用 | gemma3:4b
服务可用 False | 请求真实调用 True → 离线演示（自动降级） | moondream
服务可用 False | 请求真实调用 False → 离线演示 | gemma3:4b
```

### 15.5.5 实操步骤（上机跟随）

1. 运行 15.5.4 主示例，把样本 2 补上 `vehicle_count`，确认转为「合格」。
2. 把样本 1 的 `congestion` 改成 `"严重拥堵"`，确认输出「取值非法」。
3. 运行变式代码，补上第四种组合 `(True, False)`，预测并验证输出。
4. 未启动 Ollama 时执行 `python3 case1_AI图片标注.py --live`，记录降级提示的原文。
5. 启动 Ollama 后再执行一次 `--live`，对比首行「运行模式」与「重试」计数。

### 15.5.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 合格率总是 100% | 失败样本被悄悄填了默认值 | 检查是否有 `.get(..., 默认值)` |
| 重试计数一直为 0 | 当前是演示模式 | 重试只在 `--live` 下触发 |
| 降级后输出与演示模式不同 | 模式字符串不同但数据相同 | 属预期，看「不合格样本」清单是否一致 |
| 日志只有「失败」没有原因 | 没打印 `msg` | 打印 `validate` 返回的说明 |
| 服务时好时坏 | 端口占用 / 模型反复加载 | `ollama ps` 看状态，保持 `serve` 窗口 |

---

## 15.6 知识点 15.6　RTX 3050 显存管理与自动降级

### 15.6.1 核心概念

**同名显卡，不同显存**：RTX 3050 有台式 8GB、笔记本 4GB / 6GB 多种版本。`nvidia-smi` 看到的总量决定能加载哪个模型；装不下时 Ollama 会把部分层放到 CPU 共享内存，速度可能慢 5—20 倍。

| 档位 | 视觉模型 | 文本模型 | 兜底 |
|---|---|---|---|
| 8GB | `gemma3:4b`（≈3.0—3.3 GB） | `qwen2.5:3b`（≈1.9 GB） | `moondream`（≈1.7 GB） |
| 4GB / 6GB | `moondream` | `qwen2.5:3b` | `granite3.2-vision:2b` |

**降级链**：8GB 档若显存仍紧张，可按 `gemma3:4b → qwen2.5vl:3b → moondream` 依次换小；4GB 档只有 `moondream → granite3.2-vision:2b` 两级。案例通过 `--vram 4` 直接指定档位，`select_model()` 内部用 `8 if vram_gb >= 8 else 4` 做判定，6GB 归入 4GB 档。

**原理补充（量化与显存）**：Ollama 分发的是 4-bit 量化权重（Q4），把 16 位浮点压到约 1/4，才让 3B—4B 模型进得了 8GB。但推理时除了权重，还要留出 KV cache（与上下文长度、图片分辨率相关），所以「模型 3.3GB + 8GB 显存」并不等于「一定宽裕」——大图、长提示词都会推高占用。这也是 15.3 建议把图片长边缩到 512—768 px 的深层原因。

### 15.6.2 语法要点

```python
PROFILES = {
    8: {"vision": "gemma3:4b", "text": "qwen2.5:3b"},
    4: {"vision": "moondream",  "text": "qwen2.5:3b"},
}
key = 8 if vram_gb >= 8 else 4
model = PROFILES[key][kind]
```

- 档位判定**只在一处**（`ai_common.select_model`），两个案例共用；
- 命令行 `--vram 4` 由 `parse_args()` 解析，写错或非数字一律回落到 8；
- 「本机配置卡」6 个字段（demo15-6b）是 14.9 部署实验的直接产出。

### 15.6.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 4GB 机器跑 `gemma3:4b` | 显存溢出或极慢 | `--vram 4` |
| `--vram4`（漏空格） | 参数没生效，仍用 8GB 档 | 写 `--vram 4` |
| 代码模型名与 `ollama list` 不一致 | `model not found` | 连 tag 一起复制 |
| 浏览器 / 游戏占着显存 | 可用显存不足 | 关掉再跑，`nvidia-smi` 复核 |
| 选 11B / 13B 视觉模型 | 单次推理几分钟 | 不在候选清单内 |

### 15.6.4 随堂示例：档位与降级链

In [ ]:
CHAIN = {
    8: ["gemma3:4b", "qwen2.5vl:3b", "moondream"],
    4: ["moondream", "granite3.2-vision:2b"],
}

def chain(vram):
    key = 8 if vram >= 8 else 4
    return key, CHAIN[key]

for v in (4, 6, 8):
    key, models = chain(v)
    print(v, "GB →", key, "GB 档 | 首选", models[0], "| 兜底", models[-1], "| 候选数", len(models))

**预期输出**：

```text
4 GB → 4 GB 档 | 首选 moondream | 兜底 granite3.2-vision:2b | 候选数 2
6 GB → 4 GB 档 | 首选 moondream | 兜底 granite3.2-vision:2b | 候选数 2
8 GB → 8 GB 档 | 首选 gemma3:4b | 兜底 moondream | 候选数 3
```

**变式代码（迁移练习）**：生成 14.9 要求的「本机配置卡」，部署实验时把真实值填进去。

In [ ]:
card = {
    "gpu": "RTX 3050",
    "vram_gb": 8,
    "vision_model": "gemma3:4b",
    "text_model": "qwen2.5:3b",
    "endpoint": "127.0.0.1:11434",
    "offline_infer": True,
}
for k, v in card.items():
    print(k, "：", v)
print("配置卡字段数：", len(card))
print("推理可断网：", card["offline_infer"])

**预期输出**：

```text
gpu ： RTX 3050
vram_gb ： 8
vision_model ： gemma3:4b
text_model ： qwen2.5:3b
endpoint ： 127.0.0.1:11434
offline_infer ： True
配置卡字段数： 6
推理可断网： True
```

### 15.6.5 实操步骤（上机跟随）

1. 运行 15.6.4 主示例，把 `chain(6)` 之外的调用换成 `chain(12)`，预测档位。
2. 按 14.9 步骤 1 记录本机 `nvidia-smi` 显存，把 demo15-6b 的 `vram_gb` 改成实测值，重新打印配置卡。
3. 执行 `python3 case1_AI图片标注.py --live --vram 4`，确认首行模型是 `moondream`。
4. 执行 `python3 case2_交通事件NLP.py --live --vram 4`，确认模型仍是 `qwen2.5:3b`（文本模型不随档位变）。
5. 真实推理时另开终端 `nvidia-smi` 与 `ollama ps`，记录峰值显存与已加载模型名。

### 15.6.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `CUDA error: out of memory` | 权重 + KV cache 超显存 | 换更小模型、缩图、关占显存程序 |
| 推理慢到几分钟 | 退到 CPU | `ollama ps` 看 PROCESSOR 列 |
| `--vram 4` 没生效 | 参数写法不对 | 必须是 `--vram 4`，空格分开 |
| `model not found` | 该档模型没 pull | 按 15.0.1 对应档位补拉 |
| 首次推理特别慢 | 权重从磁盘加载进显存 | 属正常，第二次会明显变快 |
| 同一显卡同学结果不同 | 显存版本 / 驱动 / 模型版本不同 | 以配置卡为准，不横向比对耗时 |

---

# 第二部分　代码示范

## 案例 15-1　基于 Python 的 AI 图片标注实现

### 任务情境

区级交通运行中心每天筛上千张卡口图，先挑畅通、雾天、事故、拥堵四类试点。值班员要在离线演示模式下先跑一遍：逐张给出场景、拥堵等级、车辆数、风险等级与描述，合格结果落成 `labels.json` 与 CSV，并处理 `road_foggy.png` 这种判读失败的图。

**任务**：批量读取交通场景图片，用本机视觉模型逐张标注「场景 / 拥堵等级 / 车辆数 / 风险等级 / 描述」，把合格结果写成 `labels.json` 与 `labels.csv`。

**涉及知识点**：15.1 / 15.2 / 15.3 / 15.5 / 15.6，并复用第 14 章 `pathlib`、`base64`、`json`、`csv` 与第 9 章容错。

**运行前置**：默认离线演示模式无需任何环境；`--live` 需按 15.0.1 完成部署并 `ollama pull gemma3:4b`（4GB 档为 `moondream`）。

### 输入规格

| 项 | 要求 |
|---|---|
| 目录 | 默认读本章目录下的 `sample_images/`；不存在或无 PNG 时，由 `make_sample_images.build()` 在临时目录现造 4 张 64×64 示意 PNG |
| 文件 | `road_clear.png`、`road_foggy.png`、`road_incident.png`、`road_jam.png`，按 `sorted()` 后顺序处理 |
| 请求体字段 | `model`、`messages`、`stream`、`format`、`options`；图片放在 `messages` 首条的 `images` 列表，值为纯 base64 字符串，不带 data URL 前缀 |
| 取值 | `format.required` 为 `scene`、`congestion`、`vehicle_count`、`risk`、`caption`；`congestion` 三选一、`risk` 四选一、`vehicle_count` 为整数；`--vram` 大于等于 8 归 8 档，否则归 4 档 |
| 非法值 | `--vram abc` 回落 8 档；`--vram 0` 归 4 档并换 `moondream`；`--live` 但服务未启时自动降级；`images` 传 data URL 得到 `HTTPError: 400` |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 离线演示模式终端恰好 15 行：1 行模式 + 1 行图片数 + 4 行逐图结论 + 1 行不合格清单 + 2 行落盘 + 1 行已知答案 + 5 行汇总 |
| 逐行格式 | 逐图结论行以两个空格开头，形如 `  road_clear.png → 校验通过`；失败行追加「（记入失败日志）」；汇总以 `—— 运行汇总 ——` 开头 |
| 输出字段 | `labels.json` 每条键为 `scene`、`congestion`、`vehicle_count`、`risk`、`caption`、`file`、`model`；`labels.csv` 列序为 `file`、`scene`、`congestion`、`vehicle_count`、`risk`、`caption`、`model`，编码 `utf-8-sig` |
| 精度 / 单位 | 合格率保留 1 位小数，离线演示为 75.0 %；`vehicle_count` 与命中数为整数 |
| 判定方式 | 默认模式与「运行结果」逐字比对 15 行；再按 15.0.4 验证 2 读产物文件，条数为 3 且字段齐全 |

In [ ]:
# -*- coding: utf-8 -*-
"""案例 15-1：基于本地 Ollama 的交通场景图片批量标注（RTX 3050 适配）。

用法：
  python3 case1_AI图片标注.py                    # 离线演示模式（默认，无需 Ollama / GPU）
  python3 case1_AI图片标注.py --live             # 真实调用本机 Ollama
  python3 case1_AI图片标注.py --live --vram 4    # 4GB/6GB 显存档位，自动换 moondream

输出：output/labels.json、output/labels.csv
离线边界：真实调用只连 127.0.0.1:11434；首次 ollama pull 需联网，之后可断网运行。
"""
import json
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

from ai_common import (check_service, image_to_b64, post_chat, save_csv,
                       save_json, select_model, summary, validate)
from make_sample_images import build as build_images

HERE = Path(__file__).resolve().parent
OUT = HERE / "output"

SCHEMA = {
    "type": "object",
    "properties": {
        "scene": {"type": "string"},
        "congestion": {"type": "string"},
        "vehicle_count": {"type": "integer"},
        "risk": {"type": "string"},
        "caption": {"type": "string"},
    },
    "required": ["scene", "congestion", "vehicle_count", "risk", "caption"],
}
REQUIRED = SCHEMA["required"]
ENUMS = {
    "congestion": ["畅通", "缓行", "拥堵"],
    "risk": ["无", "低", "中", "高"],
}
FIELDS = ["file", "scene", "congestion", "vehicle_count", "risk", "caption", "model"]
PROMPT = (
    "你是交通场景标注员。只输出一个 JSON 对象，不要任何解释文字。"
    "字段：scene（场景简述）、congestion（畅通/缓行/拥堵三选一）、"
    "vehicle_count（整数，画面中的车辆数）、risk（无/低/中/高四选一）、"
    "caption（一句中文描述）。"
)

# 离线演示用的确定性桩：前三张是「已知答案样本」，用于效果验证
DEMO_ANSWERS = {
    "road_clear": {"scene": "城市道路", "congestion": "畅通", "vehicle_count": 1,
                   "risk": "无", "caption": "绿灯路口仅一辆车通行"},
    "road_jam": {"scene": "城市道路", "congestion": "拥堵", "vehicle_count": 11,
                 "risk": "低", "caption": "多车密集排队，通行缓慢"},
    "road_incident": {"scene": "事故现场", "congestion": "缓行", "vehicle_count": 2,
                      "risk": "高", "caption": "两车相撞并设有警示标志"},
}
KNOWN = [
    ("road_clear", "congestion", "畅通"),
    ("road_jam", "congestion", "拥堵"),
    ("road_incident", "risk", "高"),
]


def demo_answer(stem):
    """演示桩：road_foggy 故意返回非 JSON 文本，用来走通容错分支。"""
    if stem == "road_foggy":
        return "这张图太模糊了，我无法判断，建议重新拍摄。"
    return json.dumps(DEMO_ANSWERS.get(stem, {}), ensure_ascii=False)


def infer_content(path, model, live, strict=False):
    if not live:
        return demo_answer(path.stem)
    prompt = PROMPT if not strict else PROMPT + " 请严格只输出 JSON，不要任何前后缀。"
    return post_chat(model, prompt, SCHEMA, images=[image_to_b64(path)])


def parse_args(argv):
    live = "--live" in argv
    vram = 8
    if "--vram" in argv:
        i = argv.index("--vram")
        if i + 1 < len(argv):
            try:
                vram = int(argv[i + 1])
            except ValueError:
                vram = 8
    return live, vram


def verify_output(rows, known):
    """已知答案样本核对：真实模型上线后，这个数字就是持续监控的质量指标。"""
    hits = 0
    for stem, field, expect in known:
        row = next((r for r in rows if r["file"] == stem + ".png"), None)
        if row and row[field] == expect:
            hits += 1
    print("已知答案样本： %d 条 | 命中： %d" % (len(known), hits))
    return hits


def main(argv=None):
    argv = sys.argv[1:] if argv is None else argv
    live, vram = parse_args(argv)
    vram_key, model = select_model("vision", vram)
    mode = "真实调用" if live else "离线演示"

    if live:
        ok, info = check_service()
        if not ok:
            print("本机 Ollama 未就绪（%s），自动降级为离线演示模式" % info)
            live = False
            mode = "离线演示（自动降级）"

    print("运行模式：", mode, "| 显存档位：", vram_key, "GB | 模型：", model)

    with TemporaryDirectory() as tmp:
        img_dir = HERE / "sample_images"
        imgs = sorted(img_dir.glob("*.png")) if img_dir.exists() else []
        if not imgs:                       # 没有自备图片时，用标准库造 4 张示意图
            img_dir = Path(tmp)
            build_images(img_dir)
            imgs = sorted(img_dir.glob("*.png"))
        print("待标注图片：", len(imgs), "张")

        rows, failed, retries = [], [], 0
        for p in imgs:
            content = infer_content(p, model, live)
            data, msg = validate(content, REQUIRED, ENUMS)
            if data is None and live:      # 真实调用失败时，收紧提示词重试一次
                content = infer_content(p, model, live, strict=True)
                data, msg = validate(content, REQUIRED, ENUMS)
                retries += 1
            if data is None:
                failed.append(p.name)
                print(" ", p.name, "→", msg, "（记入失败日志）")
                continue
            print(" ", p.name, "→", msg)
            row = dict(data)
            row["file"] = p.name
            row["model"] = model
            rows.append(row)

        if failed:
            print("不合格样本：", failed)
        save_json(rows, OUT / "labels.json")
        save_csv(rows, OUT / "labels.csv", FIELDS)
        print("已写入 output/labels.json（%d 条）" % len(rows))
        print("已写入 output/labels.csv（%d 条）" % len(rows))
        verify_output(rows, KNOWN)

    summary(len(rows), len(imgs), retries, mode, model, vram_key)


if __name__ == "__main__":
    main()

### 运行结果（默认离线演示模式）

```text
运行模式： 离线演示 | 显存档位： 8 GB | 模型： gemma3:4b
待标注图片： 4 张
  road_clear.png → 校验通过
  road_foggy.png → JSON 解析失败：Expecting value （记入失败日志）
  road_incident.png → 校验通过
  road_jam.png → 校验通过
不合格样本： ['road_foggy.png']
已写入 output/labels.json（3 条）
已写入 output/labels.csv（3 条）
已知答案样本： 3 条 | 命中： 3
—— 运行汇总 ——
运行模式： 离线演示 | 显存档位： 8 GB | 模型： gemma3:4b
处理条数： 4 | 合格： 3 | 重试： 0
合格率： 75.0 %
服务地址： 127.0.0.1:11434（仅本机）| 外部在线 API： 无
```

### 解析

1. **两张图两种命运**：`road_clear / road_incident / road_jam` 是已知答案样本，走 `DEMO_ANSWERS` 得到合法 JSON；`road_foggy` 故意返回一句自然语言，触发 `json.JSONDecodeError`。这条分支不是 bug，是**专门设计的容错演示**。
2. **失败不中断**：`validate()` 返回 `(None, 原因)`，主程序 `failed.append(p.name)` 后 `continue`，其余 3 张照常写入。合格率 3/4 = 75.0%，分母是**全部 4 张**而不是合格张数。
3. **演示与真实共用一条校验链**：`infer_content()` 只负责「拿到 content 字符串」，来源可以是桩也可以是 `post_chat`；后面的 `validate / 落盘 / 汇总` 完全一样。这样学生看懂演示模式，就等于看懂了真实模式。
4. **重试只在真实调用时发生**：`if data is None and live` 收紧提示词再试一次，`retries` 计数进入汇总。演示模式数据确定，重试没有意义。
5. **两个输出文件**：JSON 供程序继续处理（`indent=2` 便于人看），CSV 供 Excel 打开（`utf-8-sig` 防中文乱码）。两者都只含 3 条合格记录，坏样本出现在日志里。
6. **`verify_output()` 是效果验证的核心**：用 3 条人工标注的已知答案核对 `congestion` / `risk` 字段，打印命中数。换成真实模型后，这个数字就是持续监控的质量指标。

### 切换到真实模型

```bash
ollama serve                              # 另开终端保持运行
python3 case1_AI图片标注.py --live         # 8GB 档
python3 case1_AI图片标注.py --live --vram 4  # 4GB/6GB 档
```

输出格式不变，变化的是：首行模式变「真实调用」、模型名按档位变化、合格率与命中数由模型实际表现决定、耗时明显变长。若服务未启动，会打印「本机 Ollama 未就绪……自动降级为离线演示模式」并继续运行。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| `model not found` | 模型没 pull 或名称漏 tag | `ollama list` 逐字核对 |
| 4 张图全失败 | `images` 传错 / 模型不支持视觉 | 确认用 `gemma3:4b` 或 `moondream`，不是纯文本模型 |
| `Connection refused` | 服务未启 | `ollama serve` 或接受自动降级 |
| `output/` 找不到 | 在项目根目录之外执行 | 回到 `15本地AI综合实战/` 目录 |
| 推理极慢 | 原图过大 / 显存不足 | 缩图；`--vram 4` 换 `moondream` |
| CSV 打开乱码 | 用非 UTF-8 编辑器 | 用 Excel 打开，`utf-8-sig` 已处理 |

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| `road_foggy.png` 也显示「校验通过」 | 失败：容错分支未生效 | 查 `demo_answer()` 是否返回非 JSON，`validate()` 是否返回 `None` |
| 「不合格样本：」打印空列表 | 失败：坏样本被填了默认值 | 查 `if data is None` 分支是否 `failed.append(...)` 后 `continue` |
| 汇总合格率显示 100.0 % | 失败：分母用了合格条数 | 核对 `summary()` 实参，`total` 应为 4 |
| `labels.json` 中文变成转义码 | 失败：序列化缺 `ensure_ascii=False` | 查 `save_json()` 内的序列化参数 |
| 报错 `model not found` | 失败：模型未拉取或漏 tag | 执行 `ollama list` 核对模型名 |
| 报错 `Connection refused` 且未降级 | 失败：`check_service()` 没被调用 | 查 `if live:` 分支与降级赋值 |
| 报错 `FileNotFoundError: output/...` | 失败：不在本章目录执行 | 回到 `15本地AI综合实战/` 再运行 |

---

## 案例 15-2　基于 Python 的自然语言处理实现

### 任务情境

值班日志还是自由文本，5 条记录里混着时间、车牌、路段与车速，人工归档既慢又易漏。你要先做车牌脱敏、抽出时间与路段，再按「事件类型、严重度、摘要」整理成结构化结果，与 `TRUTH` 比对准确率，最后输出 `nlp_results.json` 与 `nlp_report.csv`。

**任务**：读取 5 条交通事件自由文本，先正则脱敏与字段预抽取，再用本机文本模型判定「事件类型 / 严重度 / 摘要」，输出分布统计、准确率与两份结果文件。

**涉及知识点**：15.1 / 15.2 / 15.4 / 15.5，并复用第 14 章 `re`、`csv`、`Counter` 与第 9 章容错。

**运行前置**：默认离线演示模式无需任何环境；`--live` 需 `ollama pull qwen2.5:3b`（两种档位通用）。

### 输入规格

| 项 | 要求 |
|---|---|
| 目录与文件 | 与脚本同目录的 `events_sample.txt`，UTF-8，5 行自由文本；按行 `strip()`，空行过滤 |
| 类型 | 每行为 `str`；`preprocess()` 返回字典，含 `time`、`road`、`speed`、`text`、`plates`，其中 `speed` 为 `int`、`plates` 为车牌个数 |
| 取值范围 | 时间为 08:12、09:05、10:30、14:20、17:45；车牌每行 1 个共 5 个，全部替换为占位符；路段取自 `机场北`、`机场南`、`京开`、`京港澳`；车速 18—65 km/h |
| 请求体字段 | `model`、`messages`、`stream`、`format`、`options`；`format.required` 为 `event_type`、`road`、`severity`、`time`、`summary` |
| 取值 | `event_type` 四选一（事故、拥堵、施工、通行正常）；`severity` 三选一（低、中、高）；`TRUTH` 以时间为键 |
| 非法值 | `events_sample.txt` 缺失抛 `FileNotFoundError`；路段写成 `京通快速` 时 `road` 为空串；枚举外等级被 `validate()` 拦下 |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 离线演示模式终端恰好 16 行：2 行环境 + 5 行事件结论 + 1 行分布 + 1 行准确率 + 2 行落盘 + 5 行汇总 |
| 逐行格式 | 事件结论行以两个空格开头，形如 `  事件 1 → 事故 / 高 / 机场北`，三段用「空格 斜杠 空格」分隔 |
| 输出字段 | `nlp_results.json` 每条键为 `event_type`、`road`、`severity`、`time`、`summary`、`speed`、`model`；`nlp_report.csv` 列序为 `time`、`road`、`event_type`、`severity`、`speed`、`summary`、`model` |
| 精度 / 单位 | 准确率保留 1 位小数，离线演示为 100.0 %；`speed` 为整数，单位 km/h |
| 判定方式 | 与「运行结果」逐字比对 16 行；`output/nlp_results.json` 为 5 条，`nlp_report.csv` 为 1 行表头加 5 行数据 |

In [ ]:
# -*- coding: utf-8 -*-
"""案例 15-2：基于本地 Ollama 的交通事件文本结构化抽取与分类（RTX 3050 适配）。

用法：
  python3 case2_交通事件NLP.py                    # 离线演示模式（默认）
  python3 case2_交通事件NLP.py --live             # 真实调用本机 Ollama
  python3 case2_交通事件NLP.py --live --vram 4    # 4GB/6GB 显存档位

输入：events_sample.txt（5 条交通事件自由文本）
输出：output/nlp_results.json、output/nlp_report.csv
"""
import json
import re
import sys
from collections import Counter
from pathlib import Path

from ai_common import (check_service, post_chat, save_csv, save_json,
                       select_model, summary, validate)

HERE = Path(__file__).resolve().parent
OUT = HERE / "output"
EVENTS_FILE = HERE / "events_sample.txt"

PLATE = re.compile(r"[\u4e00-\u9fa5][A-Z]\d{5}")
ROAD = re.compile(r"机场北|机场南|京开|京港澳")
TIME = re.compile(r"\d{2}:\d{2}")
SPEED = re.compile(r"(\d+)\s*km/h")

SCHEMA = {
    "type": "object",
    "properties": {
        "event_type": {"type": "string"},
        "road": {"type": "string"},
        "severity": {"type": "string"},
        "time": {"type": "string"},
        "summary": {"type": "string"},
    },
    "required": ["event_type", "road", "severity", "time", "summary"],
}
REQUIRED = SCHEMA["required"]
ENUMS = {
    "event_type": ["事故", "拥堵", "施工", "通行正常"],
    "severity": ["低", "中", "高"],
}
FIELDS = ["time", "road", "event_type", "severity", "speed", "summary", "model"]
PROMPT = (
    "你是交通事件分类员。只输出一个 JSON 对象，不要解释。"
    "字段：event_type（事故/拥堵/施工/通行正常四选一）、road（路段名）、"
    "severity（低/中/高三选一）、time（HH:MM）、summary（一句中文摘要）。"
)
TRUTH = {
    "08:12": "事故",
    "09:05": "拥堵",
    "10:30": "事故",
    "14:20": "施工",
    "17:45": "通行正常",
}


def preprocess(line):
    """正则预清洗：车牌脱敏 + 抽出时间 / 路段 / 车速，减少模型负担。"""
    plates = len(PLATE.findall(line))
    clean = PLATE.sub("【车牌】", line)
    t, road, sp = TIME.search(clean), ROAD.search(clean), SPEED.search(clean)
    return {
        "time": t.group() if t else "",
        "road": road.group() if road else "",
        "speed": int(sp.group(1)) if sp else 0,
        "text": clean,
        "plates": plates,
    }


def demo_answer(item):
    """离线演示桩：按关键词规则给出确定性分类，走与真实模型相同的校验链路。"""
    text = item["text"]
    if "追尾" in text or "撞" in text:
        et, sev = "事故", "高"
    elif "施工" in text:
        et, sev = "施工", "中"
    elif "缓行" in text or "通行缓慢" in text:
        et, sev = "拥堵", "中"
    else:
        et, sev = "通行正常", "低"
    return json.dumps({
        "event_type": et,
        "road": item["road"],
        "severity": sev,
        "time": item["time"],
        "summary": text[:18],
    }, ensure_ascii=False)


def infer_content(item, model, live, strict=False):
    if not live:
        return demo_answer(item)
    prompt = PROMPT + "\n待分类文本：" + item["text"]
    if strict:
        prompt += "\n请严格只输出 JSON，不要任何前后缀。"
    return post_chat(model, prompt, SCHEMA)


def parse_args(argv):
    live = "--live" in argv
    vram = 8
    if "--vram" in argv:
        i = argv.index("--vram")
        if i + 1 < len(argv):
            try:
                vram = int(argv[i + 1])
            except ValueError:
                vram = 8
    return live, vram


def main(argv=None):
    argv = sys.argv[1:] if argv is None else argv
    live, vram = parse_args(argv)
    vram_key, model = select_model("text", vram)
    mode = "真实调用" if live else "离线演示"

    if live:
        ok, info = check_service()
        if not ok:
            print("本机 Ollama 未就绪（%s），自动降级为离线演示模式" % info)
            live = False
            mode = "离线演示（自动降级）"

    lines = [l.strip() for l in EVENTS_FILE.read_text(encoding="utf-8").splitlines() if l.strip()]
    items = [preprocess(l) for l in lines]
    print("运行模式：", mode, "| 显存档位：", vram_key, "GB | 模型：", model)
    print("读入事件：", len(items), "条 | 脱敏车牌：", sum(it["plates"] for it in items), "处")

    rows, failed, retries = [], [], 0
    for i, item in enumerate(items, 1):
        content = infer_content(item, model, live)
        data, msg = validate(content, REQUIRED, ENUMS)
        if data is None and live:
            content = infer_content(item, model, live, strict=True)
            data, msg = validate(content, REQUIRED, ENUMS)
            retries += 1
        if data is None:
            failed.append(i)
            print("  事件", i, "→", msg, "（记入失败日志）")
            continue
        row = dict(data)
        row["speed"] = item["speed"]
        row["model"] = model
        rows.append(row)
        print("  事件", i, "→", row["event_type"], "/", row["severity"], "/", row["road"])

    dist = Counter(r["event_type"] for r in rows)
    print("事件类型分布：", dict(dist))
    hits = sum(1 for r in rows if TRUTH.get(r["time"]) == r["event_type"])
    print("已知答案样本： %d 条 | 命中： %d | 准确率： %.1f %%"
          % (len(TRUTH), hits, hits / len(TRUTH) * 100))
    save_json(rows, OUT / "nlp_results.json")
    save_csv(rows, OUT / "nlp_report.csv", FIELDS)
    print("已写入 output/nlp_results.json（%d 条）" % len(rows))
    print("已写入 output/nlp_report.csv（%d 条）" % len(rows))

    summary(len(rows), len(items), retries, mode, model, vram_key)


if __name__ == "__main__":
    main()

### 运行结果（默认离线演示模式）

```text
运行模式： 离线演示 | 显存档位： 8 GB | 模型： qwen2.5:3b
读入事件： 5 条 | 脱敏车牌： 5 处
  事件 1 → 事故 / 高 / 机场北
  事件 2 → 拥堵 / 中 / 京开
  事件 3 → 事故 / 高 / 机场南
  事件 4 → 施工 / 中 / 京港澳
  事件 5 → 通行正常 / 低 / 机场北
事件类型分布： {'事故': 2, '拥堵': 1, '施工': 1, '通行正常': 1}
已知答案样本： 5 条 | 命中： 5 | 准确率： 100.0 %
已写入 output/nlp_results.json（5 条）
已写入 output/nlp_report.csv（5 条）
—— 运行汇总 ——
运行模式： 离线演示 | 显存档位： 8 GB | 模型： qwen2.5:3b
处理条数： 5 | 合格： 5 | 重试： 0
合格率： 100.0 %
服务地址： 127.0.0.1:11434（仅本机）| 外部在线 API： 无
```

### 解析

1. **预清洗先行**：`preprocess()` 用 `PLATE.sub` 把 5 个车牌换成「【车牌】」，再用 `TIME / ROAD / SPEED` 抽出结构化字段。模型看到的文本更干净、任务更聚焦。
2. **分类优先级有讲究**：`demo_answer()` 先判「追尾 / 撞」→ 事故，再判「施工」，再判「缓行」。第 4 条含「通行缓慢」但也含「施工」，按规则归入「施工」，说明**规则顺序会影响结果**，写真实提示词时同样要明确优先级。
3. **`TRUTH` 用时间做键**：`{"08:12": "事故", ...}`，增删一条事件不会让整张答案表错位。命中 5/5 得 100.0%——演示桩是确定性的，这个数字只用于验证流水线，不代表模型能力。
4. **分布与准确率是两件事**：`Counter` 统计「预测了哪几类、各几条」，`TRUTH` 比对「预测得对不对」。练习 15-2 专门练这两个口径，别把它们混为一个数。
5. **报表合并了机器与模型字段**：`speed` 来自正则、`event_type` 来自模型、`model` 记录溯源，形成一份可直接用于教学分析的完整 CSV。
6. **真实运行时准确率会低于 100%**：这是正常且有价值的结果。应记录「哪条判错 → 为什么 → 提示词怎么改」，再重跑看命中数是否回升，这正是回归测试集的意义。

### 切换到真实模型

```bash
python3 case2_交通事件NLP.py --live
python3 case2_交通事件NLP.py --live --vram 4
```

文本模型 `qwen2.5:3b` 在 4GB / 8GB 档都可用，`--vram` 只改变汇总里显示的档位，不会换模型。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| `road` 有空值 | `ROAD` 正则未收录该路段 | 扩充候选项 |
| 准确率突然掉到 60% | 提示词被改 / 模型版本变化 | 用 `TRUTH` 回归集定位到具体条目 |
| 车牌出现在日志 | 用了原始 `line` 而非 `clean` | 全流程统一用脱敏后文本 |
| `events_sample.txt` 读不到 | 路径或编码不对 | 在项目目录执行，确认 UTF-8 |
| 分类出现枚举外的值 | `format` 只管语法 | `ENUMS` 校验会拦下并记入失败 |
| 推理很慢 | 首次加载权重 | 属正常，第二次明显加快 |

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 事件 4 被判为「拥堵」而非「施工」 | 失败：分类优先级顺序错 | 查 `施工` 分支是否排在 `缓行`、`通行缓慢` 之前 |
| 日志里出现真实车牌 | 失败：用了未脱敏文本 | 查 `text` 是否取自 `PLATE.sub` 之后的字符串 |
| 「事件类型分布：」只有 1 类 | 失败：统计字段名写错 | 查 `Counter(r["event_type"] for r in rows)` |
| 有判错时准确率仍为 100.0 % | 失败：分母或 `TRUTH` 键写错 | 核对 `hits / len(TRUTH)` |
| `road` 为空 | 失败：`ROAD` 候选未覆盖原文路段 | 逐个对照四个候选与 5 行原文 |
| 报错 `FileNotFoundError: events_sample.txt` | 失败：工作目录不对 | 回到 `15本地AI综合实战/` 执行 |

---

# 第三部分　布置练习

## 练习 15-1　标注结果置信度过滤

- **对应知识点**：15.5（结果校验、重试与降级）
- **提交文件**：`practice1_置信度过滤.py`（UTF-8，附一次运行截图，含修正前后输出）
- **难度**：基础　**建议用时**：20 分钟

### 任务情境

交付前要先过置信度闸门：6 条记录里 `f.png` 恰好是 0.70 的边界样本，但脚本用了严格大于，把它与 `c.png`、`e.png` 一并剔除，合格率还被算成 100.0 %。你要改成大于等于、剔除条件同步改小于，分母换成 `len(ROWS)`，得到保留 4 条、合格率 66.7 %。

### 题目

`practice1_置信度过滤.py` 已能运行，但有 **2 处统计口径问题**。请修正，使输出与答案 15-1 一致。

### 要求

1. 过滤口径是**保留置信度 ≥ 0.70**（含边界），当前写成了 `> 0.70`，导致 `f.png`（正好 0.70）被误剔除；剔除清单的判断符号要同步修改；
2. 「合格率」指**保留记录占总记录的比例**，分母必须是 `len(ROWS)`；
3. 保持「总记录」「保留等级分布」等其他输出行的计算逻辑不变；
4. 只改这 2 处，不改动 `ROWS` 数据。

### 提示

0.70 是浮点数边界，别用 `==` 去判断；大于等于用 `>=`、小于用 `<` 成对出现，与 05A 节练过的边界 bug 同源。

### 自检要点（不含答案）

- [ ] 「保留记录」输出为 `4`，判定方法：`a.png`、`b.png`、`d.png`、`f.png` 四条均满足 `>= 0.70`；
- [ ] 「剔除记录」输出为 `['c.png', 'e.png']`，判定方法：剔除条件必须写成 `< THRESHOLD`；
- [ ] 「合格率」输出为 `66.7 %`，判定方法：`len(kept) / len(ROWS) * 100`，即 4 / 6；
- [ ] 「保留等级分布」输出为拥堵 1 条、缓行 2 条、畅通 1 条，且 `ROWS` 未被修改。

In [ ]:
# -*- coding: utf-8 -*-
"""练习 15-1：标注结果置信度过滤。

当前代码可以运行，但有 2 处统计口径问题，请按《案例与练习.md》练习 15-1
的题目要求修正，使输出与第四部分参考答案一致。
"""
from collections import Counter

ROWS = [
    {"file": "a.png", "level": "拥堵", "confidence": 0.93},
    {"file": "b.png", "level": "缓行", "confidence": 0.80},
    {"file": "c.png", "level": "拥堵", "confidence": 0.55},
    {"file": "d.png", "level": "畅通", "confidence": 0.72},
    {"file": "e.png", "level": "拥堵", "confidence": 0.41},
    {"file": "f.png", "level": "缓行", "confidence": 0.70},
]
THRESHOLD = 0.70

print("总记录：", len(ROWS))
# 口径问题 1：题目要求「>= 阈值」，代码写成了严格大于
print("过滤阈值： >", THRESHOLD)
kept = [r for r in ROWS if r["confidence"] > THRESHOLD]
dropped = [r["file"] for r in ROWS if r["confidence"] <= THRESHOLD]
print("保留记录：", len(kept))
print("剔除记录：", dropped)
print("保留等级分布：", dict(Counter(r["level"] for r in kept)))
# 口径问题 2：合格率分母用了保留条数，而不是总记录数
print("合格率：", round(len(kept) / len(kept) * 100, 1), "%")

### 运行结果（当前代码，尚未修正）

```text
总记录： 6
过滤阈值： > 0.7
保留记录： 3
剔除记录： ['c.png', 'e.png', 'f.png']
保留等级分布： {'拥堵': 1, '缓行': 1, '畅通': 1}
合格率： 100.0 %
```

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 边界符号成对修正 | 4 | `>=` 与 `<`，保留 4 条 |
| 合格率分母 | 3 | 4 / 6 = 66.7% |
| 其他行不变 | 2 | 分布随保留集自然变化，逻辑未改 |
| 代码注释 | 1 | 写明两处修正原因 |

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 按题目修正两处口径后运行 `practice1_置信度过滤.py` | 共 6 行；保留 4、剔除 `['c.png', 'e.png']`、合格率 66.7 % | 与答案 15-1「运行结果」6 行逐字比对 |
| T2 | 边界 | 把 `THRESHOLD` 改为 0.80 | 保留 2 条（`b.png` 恰为 0.80），剔除 4 条，合格率 33.3 % | 核对「保留记录： 2」与「合格率： 33.3 %」两行 |
| T3 | 异常 | 删掉 `ROWS` 中 `a.png` 的 `confidence` 键 | 运行即报错，退出码非 0，报错信息含 `KeyError: 'confidence'` | 看报错栈是否指向列表推导式里的取值语句 |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 边界符号成对 | 读 `kept` 与 `dropped` 两处判断符号 | `>=` 与 `<` 成对出现 | 4 |
| 合格率分母 | 读「合格率：」行 | 输出 66.7 % | 3 |
| 其他行不变 | 与答案 15-1 前 5 行逐行比对 | 总记录 6、阈值 0.7、保留等级分布与答案一致 | 2 |
| 修正说明 | 阅读源码注释 | 两处修正各有 1 行说明 | 1 |
| 合计 | — | — | 10 |

---

## 练习 15-2　事件分类分布与准确率统计

- **对应知识点**：15.4（文本预清洗与分类抽取）、15.5（结果校验与统计）
- **提交文件**：`practice2_事件分布统计.py`（UTF-8，附一次运行截图，含修正前后输出）
- **难度**：进阶　**建议用时**：25 分钟

### 任务情境

周报要交代事件分类的效果：5 条预测与人工标注逐条比对，命中 4 条，可脚本把样本数 5 当成「事件类型数」，分布一行写着「无统计」，准确率又因分母用了命中数而恒为 100.0 %。你要用 `Counter` 给出 4 个类别并按频次降序，分母换成 `len(PRED)`，其余两行不变。离线即可完成。

### 题目

`practice2_事件分布统计.py` 已能运行，但有 **2 处统计口径问题**。请修正，使输出与答案 15-2 一致。

### 要求

1. 「事件类型数」应是**去重后的类别数**，并给出按频次排序的分布，用 `collections.Counter`；
2. 「准确率」分母必须是**样本总数 `len(PRED)`**，不是命中数；
3. 保持「样本数」「命中数」两行不变，`zip(PRED, TRUTH)` 的比对逻辑不改；
4. 输出分布的形态为 `list[tuple]`，如 `[('事故', 2), ...]`。

### 提示

`Counter(PRED)` 的键数才是类别数；`len(PRED)` 是样本数。这两个数在第 14 章练习 14-1、14-2 里已经错过一次，属于同一类「不报错、只算错」的高频 bug。

### 自检要点（不含答案）

- [ ] 「事件类型数」输出为 `4`，判定方法：`Counter(PRED)` 的键为事故、拥堵、施工、通行正常；
- [ ] 「分布」输出按频次降序，含 4 个元组：事故 2、拥堵 1、施工 1、通行正常 1；
- [ ] 「准确率」输出为 `80.0 %`，判定方法：`hits / len(PRED) * 100`，即 4 / 5；
- [ ] 「样本数」仍为 `5`、「命中数」仍为 `4`，`zip` 比对逻辑未改。

In [ ]:
# -*- coding: utf-8 -*-
"""练习 15-2：事件分类分布与准确率统计。

当前代码可以运行，但有 2 处统计口径问题，请按《案例与练习.md》练习 15-2
的题目要求修正，使输出与第四部分参考答案一致。
"""
PRED = ["事故", "拥堵", "事故", "施工", "通行正常"]
TRUTH = ["事故", "拥堵", "事故", "施工", "拥堵"]

hits = sum(1 for p, t in zip(PRED, TRUTH) if p == t)
print("样本数：", len(PRED))
print("命中数：", hits)
# 口径问题 1：直接拿样本数当「事件类型数」，没有按类别去重统计
print("事件类型数：", len(PRED))
print("分布： 无统计")
# 口径问题 2：准确率分母用了命中数，恒为 100%
print("准确率：", round(hits / hits * 100, 1), "%")

### 运行结果（当前代码，尚未修正）

```text
样本数： 5
命中数： 4
事件类型数： 5
分布： 无统计
准确率： 100.0 %
```

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 用 `Counter` 去重计数 | 4 | 类别数 4，分布含 4 个元组 |
| 准确率分母 | 3 | 4 / 5 = 80.0% |
| 前两行不变 | 2 | 样本 5、命中 4 |
| 分布按频次排序 | 1 | 用 `most_common()` |

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 按题目修正两处口径后运行 `practice2_事件分布统计.py` | 共 5 行；事件类型数 4、分布 4 个元组、准确率 80.0 % | 与答案 15-2「运行结果」5 行逐字比对 |
| T2 | 边界 | 把 `TRUTH` 最后一条由 `拥堵` 改为 `通行正常` | 命中数 5、准确率 100.0 %，分布不变 | 核对「命中数： 5」与「准确率： 100.0 %」 |
| T3 | 异常 | 给 `PRED` 末尾追加一条 `未知`，`TRUTH` 不动 | 样本数 6、命中数仍 4（`zip` 截断）、事件类型数 5、准确率 66.7 % | 逐项核对这 4 个数，并确认分布新增 1 个元组 |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 去重计数 | 读「事件类型数：」与「分布：」两行 | 4 与 4 个元组，事故 2 排首位 | 4 |
| 准确率分母 | 读「准确率：」行 | 输出 80.0 % | 3 |
| 前两行不变 | 与答案 15-2 前两行比对 | 样本数 5、命中数 4 | 2 |
| 分布排序方式 | 阅读源码 | 调用 `dist.most_common()`，不是 `dict(dist)` | 1 |
| 合计 | — | — | 10 |

---

# 第四部分　习题答案

## 答案 15-1　置信度过滤（修正版）

In [ ]:
# -*- coding: utf-8 -*-
"""练习 15-1 参考答案：标注结果置信度过滤（口径修正版）。

修正点：
1. 阈值改为 >= 0.70，边界样本 f.png 应保留；剔除清单同步改为 < 阈值；
2. 合格率分母改为总记录数 len(ROWS)。
"""
from collections import Counter

ROWS = [
    {"file": "a.png", "level": "拥堵", "confidence": 0.93},
    {"file": "b.png", "level": "缓行", "confidence": 0.80},
    {"file": "c.png", "level": "拥堵", "confidence": 0.55},
    {"file": "d.png", "level": "畅通", "confidence": 0.72},
    {"file": "e.png", "level": "拥堵", "confidence": 0.41},
    {"file": "f.png", "level": "缓行", "confidence": 0.70},
]
THRESHOLD = 0.70

print("总记录：", len(ROWS))
print("过滤阈值： >=", THRESHOLD)
kept = [r for r in ROWS if r["confidence"] >= THRESHOLD]
dropped = [r["file"] for r in ROWS if r["confidence"] < THRESHOLD]
print("保留记录：", len(kept))
print("剔除记录：", dropped)
print("保留等级分布：", dict(Counter(r["level"] for r in kept)))
print("合格率：", round(len(kept) / len(ROWS) * 100, 1), "%")

### 运行结果

```text
总记录： 6
过滤阈值： >= 0.7
保留记录： 4
剔除记录： ['c.png', 'e.png']
保留等级分布： {'拥堵': 1, '缓行': 2, '畅通': 1}
合格率： 66.7 %
```

### 解析

1. **两处符号必须成对改**：`kept` 用 `>=`，`dropped` 用 `<`。只改一处会出现「既没保留也没剔除」的漏项，或同一样本同时出现在两个清单里。
2. **边界样本 `f.png`**：置信度正好 0.70，按题意保留，因此保留数 3 → 4、缓行数 1 → 2。
3. **合格率**：`4 / 6 = 66.7%`。原写法 `len(kept) / len(kept)` 恒为 1.0，是典型的「分母写错但不报错」。
4. **真实项目里的对应环节**：这就是 15.5 讲的「坏数据要记日志、不能填默认值」的后续——低置信度样本进人工复核队列，不进入正式标注集。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 保留仍是 3 条 | 只改了打印文字 | 检查列表推导式里的 `>=` |
| 剔除清单仍有 f.png | `dropped` 没同步改 `<` | 两个符号成对 |
| 合格率 100% | 分母还是 `len(kept)` | 改 `len(ROWS)` |
| 用 `== 0.70` 判断边界 | 浮点比较不可靠 | 用 `>=` / `<` 区间判断 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 6 行输出，保留 4、剔除 2、合格率 66.7 % | 正常路径 |
| T2 | 阈值改 0.80 时 `b.png` 恰好等于阈值仍被保留，保留 2 条 | 边界值落在 `>=` 一侧 |
| T3 | 缺 `confidence` 键会在列表推导式处抛 `KeyError` | 本答案不做字段容错，属预期异常 |

---

## 答案 15-2　事件分布与准确率（修正版）

In [ ]:
# -*- coding: utf-8 -*-
"""练习 15-2 参考答案：事件分类分布与准确率统计（口径修正版）。

修正点：
1. 用 collections.Counter 按事件类型去重计数，并给出分布；
2. 准确率分母改为样本总数 len(PRED)。
"""
from collections import Counter

PRED = ["事故", "拥堵", "事故", "施工", "通行正常"]
TRUTH = ["事故", "拥堵", "事故", "施工", "拥堵"]

hits = sum(1 for p, t in zip(PRED, TRUTH) if p == t)
dist = Counter(PRED)
print("样本数：", len(PRED))
print("命中数：", hits)
print("事件类型数：", len(dist))
print("分布：", dist.most_common())
print("准确率：", round(hits / len(PRED) * 100, 1), "%")

### 运行结果

```text
样本数： 5
命中数： 4
事件类型数： 4
分布： [('事故', 2), ('拥堵', 1), ('施工', 1), ('通行正常', 1)]
准确率： 80.0 %
```

### 解析

1. **样本数 ≠ 类别数**：5 条预测里「事故」出现 2 次，去重后只有 4 类。`len(Counter(PRED))` 才是类别数。
2. **`most_common()` 的顺序**：同频次按首次出现顺序，因此「拥堵」在「施工」「通行正常」之前，输出稳定可校对。
3. **准确率**：第 5 条预测「通行正常」而标准答案是「拥堵」，命中 4 条，`4 / 5 = 80.0%`。原写法 `hits / hits` 把错误样本藏了起来。
4. **与案例 15-2 的对应**：这里的 `PRED` / `TRUTH` 就是案例里 `rows` 的 `event_type` 列与 `TRUTH` 字典；真实运行时把两处替换即可复用本练习的统计代码。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 类别数仍是 5 | 直接 `len(PRED)` | `len(Counter(PRED))` |
| 分布是字典不是列表 | 用了 `dict(dist)` | `dist.most_common()` |
| 准确率 100% | 分母 `hits` | `len(PRED)` |
| 分布顺序不稳定 | 手工排序写错 | 用 `most_common()` |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 5 行输出，类别 4、准确率 80.0 % | 正常路径 |
| T2 | 第 5 条改对后命中 5，分母仍是 `len(PRED)`，得 100.0 % | 分母未写死为常量 |
| T3 | 追加 `未知` 后分母变 6，`zip` 截断使命中仍 4，准确率 66.7 % | 见解析第 3 条的计算口径 |

---

---

## 常见问题总表

| 环节 | 现象 | 根因 | 处理 |
|---|---|---|---|
| 部署 | `ollama: command not found` | 未安装 / PATH 未刷新 | 重装并重开终端 |
| 部署 | `Connection refused` | `serve` 没运行 | 保持服务窗口，或接受自动降级 |
| 部署 | 拉取到一半失败 | 网络不稳 | 重跑 `ollama pull`，支持续传 |
| 模型 | `model not found` | 名称漏 tag 或没下载 | `ollama list` 逐字复制 |
| 模型 | 推理慢到几分钟 | 显存不足退 CPU | 换 `moondream` / `qwen2.5:3b` |
| 请求 | `HTTPError: 400` | `images` 不是列表 / 字段名错 | 打印 `req.data` 核对 |
| 请求 | 程序假死 | `urlopen` 无 timeout | `ai_common.TIMEOUT = 120` 已设 |
| 解析 | `JSONDecodeError` | 模型输出解释文字 | 收紧提示词重试一次；仍失败记日志 |
| 解析 | 枚举外取值 | `format` 只约束语法 | `ENUMS` 语义校验 |
| 数据 | 合格率 / 准确率恒为 100% | 分母写成命中数 | 分母改样本总数 |
| 数据 | 类别数 = 样本数 | 忘记去重 | `Counter` 的键数 |
| 文件 | CSV 中文乱码 | 编辑器编码不对 | 用 Excel 打开（`utf-8-sig`） |
| 文件 | `output/` 找不到 | 工作目录不对 | 在章节目录执行 |
| 认知 | 把演示结果当模型能力 | 没看首行模式 | 首行「运行模式」必读 |
| 认知 | 以为断网也能 `pull` | 混淆下载与推理 | 明确两阶段边界 |

---

## 教师用：练习口径问题速查

| 练习代码 | 预设问题 | 修复要点 | 参考答案输出 |
|---|---|---|---|
| `practice1_置信度过滤.py` | ① `> 0.70` 漏掉边界样本 ② 合格率分母用保留条数 | `>=` / `<` 成对改；分母 `len(ROWS)` | 保留 4、合格率 66.7% |
| `practice2_事件分布统计.py` | ① 用样本数当类别数、无分布 ② 准确率分母用命中数 | `Counter(PRED).most_common()`；分母 `len(PRED)` | 类别 4、准确率 80.0% |

---

## 本章小结

- **两个案例、一条链路**：图片标注传 `images`，NLP 传文本，其余「选模型 → POST → 校验 → 统计 → 落盘」完全同构，共用 `ai_common.py`。
- **默认离线演示是教学设计，不是妥协**：不装 Ollama 也能看懂全流程、校对输出；`--live` 才上真模型，服务不通自动降级。
- **语法约束 ≠ 语义正确**：`format` 只保证 JSON 可解析；缺字段、枚举越界必须靠 `validate()` 拦下，坏数据记日志不填默认值。
- **三层防线**：语法层 `json.loads`、语义层 `REQUIRED + ENUMS`、服务层 `check_service()`；真实调用失败只重试一次。
- **质量要可度量**：已知答案样本命中数、合格率、准确率、重试次数、不合格清单，五个数字共同构成实验报告。
- **统计口径仍是最高频 bug**：分母写错、边界符号写错、样本数当类别数，三者都「不报错、只算错」，用 05A 节的调试器设断点看变量最快。
- **离线边界**：首次 `pull` 联网、推理可断网；`127.0.0.1:11434` 仅本机；无账号、无密钥、无外部在线 API。
- **显存决定模型**：8GB 档 `gemma3:4b` + `qwen2.5:3b`，4GB/6GB 档 `moondream` + `qwen2.5:3b`；缩图能同时省请求体与 KV cache。

## 自检清单

- [ ] 能画出一次本地 AI 调用的 7 步链路，并指出哪一步用到显卡；
- [ ] 能解释「离线运行」与「首次 pull 需联网」的区别，会做断网自检；
- [ ] 会写提示词三要素与 JSON Schema，能说明 `format` 的局限；
- [ ] 知道 `images` 必须是纯 base64 字符串列表，且大图要先缩放；
- [ ] 会用正则预清洗与脱敏，能说明「先正则、后模型」的理由；
- [ ] 能独立说出三层防线，并解释为什么重试只做一次、只在 `--live` 下做；
- [ ] 会用 `nvidia-smi` 确认显存，按 4GB / 8GB 档选模型并完成 `--vram 4` 切换；
- [ ] 能在默认模式下跑通两个案例，输出与讲义逐字一致；
- [ ] 能检查 `output/` 下 4 个结果文件的结构与字段完整性；
- [ ] 完成练习 15-1、15-2 的两处口径修正，输出与参考答案一致；
- [ ] （本机部署后）完成验证 3—5，并填好「本机配置卡」与实验报告。

## 前后章节衔接

- **第 14 章是本章的直接前置**：14.9 的部署产出「本机配置卡」在这里决定模型选型；14.2 / 14.4 / 14.5 构成请求链路，14.3 / 14.6 / 14.7 构成校验与落盘链路。
- **第 9 章异常处理**在这里升级为工程容错：`validate()` 把「模型返回坏数据」收敛成业务数据，`check_service()` 把「服务不通」提前拦下并降级，两者职责不同。
- **05A 节调试技能**在练习 15-1、15-2 再次派上用场：两题都是分母 / 边界 / 去重口径错误，建议在 `print` 前设断点看 `len(kept)`、`len(ROWS)`、`len(dist)`、`hits` 四个值。
- **第 10 章模块与程序包**：`ai_common.py` 是课程里第一个真正被多个程序复用的自建模块，可回头对照 10.x 的 `import` 与包组织规则。
- 本章之后进入**终结性考核（99）**：考核回归第 1—13 章的文本数据处理主线；第 14—15 章作为本地 AI 拓展模块，供课程设计、技能竞赛与学有余力的学生选做。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：本地AI应用前置技能](../14本地AI应用前置技能/14_本地AI应用前置技能.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：终结性考核 ▶](../99终结性考核/99_终结性考核.ipynb)